## 0 — Imports & device

In [ ]:
# !pip install torch numpy matplotlib scipy python-docx   # uncomment if not already installed
import os, re, time, csv, datetime, shutil, tempfile, warnings, hashlib, importlib, subprocess, sys, copy
import json as _json
from concurrent.futures import ProcessPoolExecutor
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
%matplotlib inline

warnings.filterwarnings('ignore')

# Some Windows runtimes need this event-loop policy for torch compatibility
import asyncio
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

IN_COLAB = 'google.colab' in sys.modules
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Environment : {"Google Colab" if IN_COLAB else "local (VS Code / Jupyter)"}')
print(f'Device      : {DEVICE.upper()}' +
      (f'  ({torch.cuda.get_device_name(0)})' if DEVICE == 'cuda' else ''))

## 1 — Configuration

Edit here, then run the rest of the notebook.

* `MODELS_TO_RUN` — which DL models to include (any subset of the four).
* `PRETRAINED_CHECKPOINTS` — set a path to **reuse** a saved checkpoint for that model
  (no training); leave `None` to train it in this run. The checkpoint's architecture
  must match the settings below.
* `FAST = True` — tiny dataset / few epochs / small eval sets for a quick smoke test.

In [ ]:
# ── System ────────────────────────────────────────────────────────────
Nt    = 32        # transmit antennas
K     = 16         # single-antenna users
L     = 5         # multipath components per user
ARRAY = 'ULA'     # 'ULA' or 'UPA'

# ── Dataset ───────────────────────────────────────────────────────────
N_TRAIN         = 200_000
N_VAL           = 10_000
N_TEST          = 20_000
BATCH_SIZE      = 512
CHUNK_SIZE      = 10_000
SNR_RANGE       = (0, 30)    # data SNR (dB), drawn uniformly per sample
PILOT_SNR_RANGE = (0, 20)    # pilot SNR (dB) -> CSI error, drawn uniformly per sample

# WMMSE pre-training labels (offline, cached with the dataset). Eval-time WMMSE in the
# plots uses the tighter n_iter=100 / tol=1e-6 default of wmmse_bf().
WMMSE_N_ITER = 30
WMMSE_TOL    = 1e-4

USE_DATASET_CACHE     = True    # reuse a cached dataset (incl. WMMSE labels) if one matches
FORCE_REGENERATE_DATA = False   # True -> ignore any existing cache and rebuild

# Dataset supplied as an uploaded zip of a `dataset_cache` folder (e.g. downloaded from
# Google Drive; multi-part downloads ...-1-001.zip, ...-1-002.zip are all picked up).
# When set, the dataset is taken ONLY from the zip -- nothing is generated, and the
# run stops with an error if the zip holds no dataset for this configuration.
# None -> use / build the cache in BASE_DIR/dataset_cache as usual.
DATASET_ZIP         = '/content/dataset_cache-*.zip' if IN_COLAB else None
DATASET_EXTRACT_DIR = '/content/dataset_cache_unzipped'

# ── Training (identical protocol for every DL model) ─────────────────
LR              = 1e-4
MAX_EPOCHS      = 100
PATIENCE        = 25
PRETRAIN_EPOCHS = 20     # MSE warm-start against the offline WMMSE target
GRAD_CLIP       = 1.0
# Per Hu et al. 2021, IAIDNN trains end-to-end on the sum-rate loss from a ZF
# initialisation -- no supervised warm start. BeamNet, BlackboxFNN and FNN-Zhang
# all use WMMSE-label pre-training (as in their source papers).
NO_PRETRAIN_MODELS = {'IAIDNN'}

# ── BeamTransformer ──────────────────────────────────────────────────
D_MODEL  = 128
N_LAYERS = 5
N_HEADS  = 4
DROPOUT  = 0.1

# ── Baselines ────────────────────────────────────────────────────────
IAIDNN_LAYERS     = 5
XIA_KERNELS       = 8
ZHANG_KERNELS     = 16

# ── Architecture ablation (BeamNet's Transformer replaced by an MLP) ──
# A) BeamNet-UserMLP: BeamNet with attention removed. Default = same width/depth
#    as BeamNet (d=128, 5 blocks -> ~0.68 M params at Nt=32, K=8). For a
#    parameter-matched version (~1.07 M, like BeamNet's 1.01 M) use USERMLP_LAYERS = 8.
USERMLP_D_MODEL = D_MODEL
USERMLP_LAYERS  = N_LAYERS
# B) FlatMLP: whole flattened channel in, W out. 3 x 512 -> ~1.05 M params at
#    Nt=32, K=8 (parameter-matched to BeamNet). Its size grows with K.
FLATMLP_HIDDEN  = 512
FLATMLP_LAYERS  = 3

# ── Which models to run, and optional checkpoints to reuse ───────────
# Any subset of: 'BeamNet', 'BeamNet-UserMLP', 'FlatMLP', 'BlackboxFNN', 'IAIDNN', 'FNN-Zhang'
# Current setting: architecture-ablation run (the two MLP variants only). For the full
# comparison use ['BeamNet', 'BlackboxFNN', 'IAIDNN', 'FNN-Zhang'] (+ the MLPs if wanted).
MODELS_TO_RUN = ['BeamNet-UserMLP', 'FlatMLP']

# Set a path to reuse a checkpoint from an earlier run with the SAME Nt/K and
# architecture settings (the dataset cache is reused automatically), e.g.
#   '/content/drive/MyDrive/BeamTransformer_Merged/Run_1/checkpoints/BeamNet_best.pt'
PRETRAINED_CHECKPOINTS = {
    'BeamNet':         None,
    'BeamNet-UserMLP': None,
    'FlatMLP':         None,
    'BlackboxFNN':     None,
    'IAIDNN':          None,
    'FNN-Zhang':       None,
}
# NOTE: checkpoints produced by the two ORIGINAL notebooks were trained on a
# mis-flattened dataset layout (see the Dataset section) -- retrain rather than reuse them.

# ── Evaluation ───────────────────────────────────────────────────────
EVAL_PILOT_SNR  = 15.0    # pilot SNR (dB) for the sum-rate / BER / antenna evals
EVAL_REF_SNR    = 20.0    # data SNR (dB) for the ablations and the "@ 20 dB" tables
N_EVAL_SUMRATE  = 1500    # channels for sum-rate vs SNR
N_EVAL_BER      = 200     # channels for BER vs SNR
N_BER_SYMBOLS   = 5000    # QPSK symbols per user per channel
N_EVAL_ABLATION = 600     # channels for antenna / CSI-quality ablations

# ── Misc ─────────────────────────────────────────────────────────────
SEED   = 0
FAST   = False    # True -> tiny dataset + few epochs (quick demo)
AUTHOR = 'Tarvir Anjum Aditto'
COURSE = 'Broadband Wireless Communication — MSc Project'

if FAST:
    print('[Fast mode] Reducing dataset, epochs and eval sizes for a quick demo.')
    N_TRAIN, N_VAL, N_TEST = 2000, 400, 400
    BATCH_SIZE, CHUNK_SIZE = 128, 1000
    MAX_EPOCHS, PATIENCE, PRETRAIN_EPOCHS = 5, 5, 3
    WMMSE_N_ITER = 15
    N_EVAL_SUMRATE, N_EVAL_BER, N_BER_SYMBOLS, N_EVAL_ABLATION = 300, 60, 2000, 200

torch.manual_seed(SEED)
SKIP_TRAINING = {n: PRETRAINED_CHECKPOINTS.get(n) is not None for n in MODELS_TO_RUN}


def save_run_params(path, extra_lines=()):
    lines = [
        f'Run ID          : {RUN_ID}',
        f'Timestamp       : {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}',
        f'Device          : {DEVICE}',
        '',
        '=== System ===',
        f'Array type      : {ARRAY}',
        f'Nt (TX antennas): {Nt}',
        f'K  (users)      : {K}',
        f'L  (paths/user) : {L}',
        '',
        '=== Dataset ===',
        f'N_TRAIN         : {N_TRAIN}',
        f'N_VAL           : {N_VAL}',
        f'N_TEST          : {N_TEST}',
        f'BATCH_SIZE      : {BATCH_SIZE}',
        f'SNR_RANGE       : {SNR_RANGE}',
        f'PILOT_SNR_RANGE : {PILOT_SNR_RANGE}',
        f'WMMSE labels    : n_iter={WMMSE_N_ITER}, tol={WMMSE_TOL}',
        '',
        '=== Training ===',
        f'LR              : {LR}',
        f'MAX_EPOCHS      : {MAX_EPOCHS}',
        f'PATIENCE        : {PATIENCE}',
        f'PRETRAIN_EPOCHS : {PRETRAIN_EPOCHS}  (not for {sorted(NO_PRETRAIN_MODELS)})',
        f'FAST mode       : {FAST}',
        '',
        '=== Models ===',
        f'MODELS_TO_RUN   : {MODELS_TO_RUN}',
        f'BeamNet         : d_model={D_MODEL}, layers={N_LAYERS}, heads={N_HEADS}, dropout={DROPOUT}',
        f'IAIDNN layers   : {IAIDNN_LAYERS}',
        f'UserMLP (abl. A): d_model={USERMLP_D_MODEL}, blocks={USERMLP_LAYERS}',
        f'FlatMLP (abl. B): hidden={FLATMLP_HIDDEN}, layers={FLATMLP_LAYERS}',
        f'Reused ckpts    : { {k: v for k, v in PRETRAINED_CHECKPOINTS.items() if v} }',
        '',
        '=== Paths ===',
        f'Run dir         : {RUN_DIR}',
        f'Checkpoints     : {CHECKPOINT_DIR}',
        f'Results         : {RESULTS_DIR}',
        f'Dataset cache   : {DATASET_CACHE_DIR}',
        f'Dataset zip     : {DATASET_ZIP}',
    ] + list(extra_lines)
    with open(path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f'[Params] Saved to {path}')

print(f'Device: {DEVICE}  |  Array: {ARRAY}  |  Nt={Nt}  |  K={K}  |  L={L}')
print(f'Models: {MODELS_TO_RUN}  |  reuse checkpoint: {[k for k, v in SKIP_TRAINING.items() if v]}')

## 2 — Run folder

* **Colab:** mounts Google Drive, `BASE_DIR = MyDrive/BeamTransformer_Merged`.
* **Local:** `BASE_DIR = LOCAL_BASE_DIR` (default: `BeamTransformer_Merged_runs/` next to
  this notebook).

Every execution creates a new `Run_N_Nt<Nt>_K<K>` folder for checkpoints, results and the
report. The dataset cache lives at `BASE_DIR/dataset_cache` (shared by all runs) and is
keyed by a hash of the dataset settings, so runs with the same system settings reuse it.

In [ ]:
LOCAL_BASE_DIR = os.path.join(os.getcwd(), 'BeamTransformer_Merged_runs')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/BeamTransformer_Merged'
else:
    BASE_DIR = LOCAL_BASE_DIR

def _next_run_id(base):
    """Next unused N among the Run_N* folders under base/."""
    ids = [int(m.group(1)) for d in (os.listdir(base) if os.path.isdir(base) else [])
           if os.path.isdir(os.path.join(base, d)) and (m := re.match(r'^Run_(\d+)', d))]
    return max(ids, default=0) + 1

os.makedirs(BASE_DIR, exist_ok=True)
RUN_ID  = _next_run_id(BASE_DIR)
RUN_DIR = os.path.join(BASE_DIR, f'Run_{RUN_ID}_Nt{Nt}_K{K}')
CHECKPOINT_DIR    = os.path.join(RUN_DIR, 'checkpoints')
RESULTS_DIR       = os.path.join(RUN_DIR, 'results')
DATASET_CACHE_DIR = os.path.join(BASE_DIR, 'dataset_cache')
for d in (CHECKPOINT_DIR, RESULTS_DIR, DATASET_CACHE_DIR):
    os.makedirs(d, exist_ok=True)

PARAMS_PATH = os.path.join(RUN_DIR, 'run_params.txt')
save_run_params(PARAMS_PATH)

print(f'Run folder     : {RUN_DIR}')
print(f'Checkpoints    : {CHECKPOINT_DIR}')
print(f'Results        : {RESULTS_DIR}')
print(f'Dataset cache  : {DATASET_CACHE_DIR}')

## 3 — Channel simulation

* **ULA** / **UPA** steering vectors (half-wavelength spacing)
* Geometry-based stochastic model: `L` paths per user within an angular spread around a
  random mean angle in ±60°; each user's channel vector is normalised to unit norm
* Imperfect CSI via pilot-noise model: `Ĥ = H + E`, `E ~ CN(0, σₑ² I)`, `σₑ² = 1/(1+SNRₚ)`

In [ ]:
# ── Steering vectors ──────────────────────────────────────────────────

def ula_steering(theta: np.ndarray, Nt: int) -> np.ndarray:
    """ULA steering vector. theta shape (...,); returns (..., Nt)."""
    n = np.arange(Nt)
    phase = np.pi * np.sin(theta)[..., np.newaxis] * n
    return (1.0 / np.sqrt(Nt)) * np.exp(1j * phase)


def upa_steering(theta: np.ndarray, phi: np.ndarray, Nv: int, Nh: int) -> np.ndarray:
    """UPA steering vector. Returns (..., Nv*Nh)."""
    av = ula_steering(theta, Nv)
    ah = ula_steering(phi,   Nh)
    a  = np.einsum('...i,...j->...ij', av, ah)
    return a.reshape(*a.shape[:-2], Nv * Nh)


# ── Channel model ─────────────────────────────────────────────────────

class MassiveMIMOChannel:
    """
    Geometry-based stochastic channel model.
    Nt: TX antennas, K: users, L: multipath per user,
    array_type: 'ULA' or 'UPA', ang_spread: degrees.
    """
    def __init__(self, Nt=32, K=8, L=5, array_type='ULA',
                 Nv=4, Nh=8, ang_spread=15.0):
        assert array_type in ('ULA', 'UPA')
        if array_type == 'UPA':
            assert Nv * Nh == Nt
        self.Nt, self.K, self.L = Nt, K, L
        self.array_type = array_type
        self.Nv, self.Nh = Nv, Nh
        self.ang_spread = np.deg2rad(ang_spread)

    def generate(self, batch_size: int, rng=None):
        """Return complex channel H of shape (batch_size, K, Nt)."""
        if rng is None:
            rng = np.random.default_rng()
        Nt, K, L, B = self.Nt, self.K, self.L, batch_size

        theta_mean = rng.uniform(-np.pi / 3, np.pi / 3, size=(B, K))
        delta = rng.uniform(-self.ang_spread / 2, self.ang_spread / 2, size=(B, K, L))
        theta = theta_mean[..., np.newaxis] + delta

        alpha = (rng.standard_normal((B, K, L)) +
                 1j * rng.standard_normal((B, K, L))) / np.sqrt(2)

        if self.array_type == 'ULA':
            a = ula_steering(theta, Nt)
        else:
            phi_mean = rng.uniform(-np.pi / 3, np.pi / 3, size=(B, K))
            phi = phi_mean[..., np.newaxis] + delta
            a   = upa_steering(theta, phi, self.Nv, self.Nh)

        H = np.einsum('bkl,bkln->bkn', alpha, a) / np.sqrt(L)
        H = H / np.sqrt(np.mean(np.abs(H)**2, axis=-1, keepdims=True) * Nt)
        return H


def make_channel(Nt_=None, K_=None):
    """Channel object with this notebook's L / array settings (Nt, K overridable)."""
    Nt_ = Nt_ or Nt
    K_  = K_  or K
    return MassiveMIMOChannel(Nt=Nt_, K=K_, L=L, array_type=ARRAY,
                              Nv=4, Nh=max(1, Nt_ // 4), ang_spread=15.0)


# ── Imperfect CSI ─────────────────────────────────────────────────────

def add_csi_noise(H: np.ndarray, pilot_snr_db: float, rng=None) -> np.ndarray:
    """Ĥ = H + E,  E ~ CN(0, sigma_e^2 I),  sigma_e^2 = 1/(1+SNR_p)."""
    if rng is None:
        rng = np.random.default_rng()
    pilot_snr_lin = 10 ** (pilot_snr_db / 10)
    sigma_e = np.sqrt(1.0 / (1.0 + pilot_snr_lin))
    noise = sigma_e * (rng.standard_normal(H.shape) +
                       1j * rng.standard_normal(H.shape)) / np.sqrt(2)
    return H + noise

## 4 — Classical beamformers & metrics

All beamformers take `H ∈ ℂ^{K×Nt}` and return a power-normalised `W ∈ ℂ^{Nt×K}`.

| Method | Idea |
|--------|------|
| **MRT** | Matched filter — max per-user SNR, ignores inter-user interference |
| **ZF-BF** | Pseudoinverse — nulls interference exactly (needs Nₜ ≥ K) |
| **MMSE-BF** | Regularised ZF (one-shot) — balances interference suppression vs. array gain |
| **WMMSE** | Iterative weighted-MMSE (Shi et al. 2011) — near-optimal sum-rate reference; also the source of the offline pre-training labels |

The cell is saved as a small module (`bt_classical.py`, next to the notebook) and then
imported. That way the process pool that builds the WMMSE labels can import these
functions — required on Windows, where worker processes cannot see functions defined
directly in a notebook.

In [ ]:
%%writefile bt_classical.py
# Auto-generated by BeamTransformer_Merged_MLPs.ipynb (Section 4) — edit the notebook, not this file.
import os
from concurrent.futures import ProcessPoolExecutor
import numpy as np


# ── Power normalisation ───────────────────────────────────────────────

def power_normalise(W: np.ndarray, P_total: float = 1.0) -> np.ndarray:
    norm = np.linalg.norm(W, 'fro')
    if norm < 1e-12:
        return W
    return W * np.sqrt(P_total) / norm


# ── MRT / ZF / MMSE ───────────────────────────────────────────────────

def mrt(H: np.ndarray, P_total: float = 1.0) -> np.ndarray:
    """Maximum Ratio Transmission:  W = H^H."""
    return power_normalise(H.conj().T, P_total)


def zf_bf(H: np.ndarray, P_total: float = 1.0) -> np.ndarray:
    """Zero-Forcing:  W = H^H (H H^H)^{-1}."""
    HHH = H @ H.conj().T
    return power_normalise(H.conj().T @ np.linalg.pinv(HHH), P_total)


def mmse_bf(H: np.ndarray, snr_db: float, P_total: float = 1.0) -> np.ndarray:
    """MMSE-BF (regularised ZF):  W = H^H (H H^H + (K/SNR) I)^{-1}."""
    K_  = H.shape[0]
    snr = 10 ** (snr_db / 10)
    HHH = H @ H.conj().T
    return power_normalise(H.conj().T @ np.linalg.inv(HHH + (K_ / snr) * np.eye(K_)), P_total)


# ── WMMSE (iterative, near-optimal reference) ─────────────────────────

def wmmse_bf(H: np.ndarray, snr_db: float, P_total: float = 1.0,
             n_iter: int = 100, tol: float = 1e-6) -> np.ndarray:
    """
    Iterative WMMSE precoder (Shi, Razaviyayn, Luo & He, IEEE TSP 2011),
    specialised to MISO (single-antenna users), so U_k, W_k are scalars.
    Alternates U -> W -> V updates to a local optimum of the weighted sum-rate,
    starting from the RZF (MMSE-BF) solution.
    """
    K_, Nt_ = H.shape
    s = 10 ** (snr_db / 10)
    sigma2 = 1.0 / s

    V = power_normalise(H.conj().T @ np.linalg.inv(H @ H.conj().T + (K_ / s) * np.eye(K_)), P_total)

    prev_rate = -np.inf
    for _ in range(n_iter):
        HV  = H @ V                                              # HV[k,m] = h_k^H v_m
        hvk = np.diag(HV)                                        # desired signal per user
        u   = hvk / (np.sum(np.abs(HV) ** 2, axis=1) + sigma2)   # MMSE receive combiner
        e   = 1.0 - u.conj() * hvk                                # residual MSE
        w   = 1.0 / np.real(e).clip(min=1e-10)                    # importance weight

        diag_term = sigma2 * np.sum(w * np.abs(u) ** 2)
        A   = diag_term * np.eye(Nt_) + (H.conj().T * (w * np.abs(u) ** 2)) @ H
        rhs = H.conj().T * (w * u)                                # columns = w_k u_k h_k
        V   = np.linalg.solve(A + 1e-10 * np.eye(Nt_), rhs)
        V   = power_normalise(V, P_total)

        rate = compute_sum_rate(H, V, snr_db)
        if abs(rate - prev_rate) < tol:
            break
        prev_rate = rate

    return V


# ── SINR & Sum-Rate ───────────────────────────────────────────────────

def compute_sinr(H: np.ndarray, W: np.ndarray, snr_db: float) -> np.ndarray:
    K_  = H.shape[0]
    snr = 10 ** (snr_db / 10)
    HW  = H @ W
    sinr = np.zeros(K_)
    for k in range(K_):
        sig   = np.abs(HW[k, k]) ** 2
        intf  = np.sum(np.abs(HW[k, :]) ** 2) - sig
        sinr[k] = sig / (intf + 1.0 / snr)
    return sinr


def compute_sum_rate(H: np.ndarray, W: np.ndarray, snr_db: float) -> float:
    return float(np.sum(np.log2(1.0 + compute_sinr(H, W, snr_db))))


# ── BER (QPSK, ZF receiver) ───────────────────────────────────────────

def compute_ber(H: np.ndarray, W: np.ndarray, snr_db: float,
                n_symbols: int = 5000, rng=None) -> float:
    """Monte-Carlo BER for QPSK using a ZF receiver on the effective channel HW."""
    if rng is None:
        rng = np.random.default_rng(0)
    K_, _ = H.shape
    snr   = 10 ** (snr_db / 10)
    bits  = rng.integers(0, 2, size=(K_, n_symbols, 2))
    syms  = (2 * bits - 1).astype(complex)
    syms_tx = (syms[..., 0] + 1j * syms[..., 1]) / np.sqrt(2)
    HW    = H @ W
    noise_std = np.sqrt(1 / (2 * snr))
    noise = noise_std * (rng.standard_normal((K_, n_symbols)) +
                         1j * rng.standard_normal((K_, n_symbols)))
    y = HW @ syms_tx + noise
    try:
        y_eq = np.linalg.solve(HW, y)
    except np.linalg.LinAlgError:
        y_eq = np.linalg.lstsq(HW, y, rcond=None)[0]
    bits_hat = np.zeros_like(bits)
    bits_hat[..., 0] = (y_eq.real > 0).astype(int)
    bits_hat[..., 1] = (y_eq.imag > 0).astype(int)
    return float(np.sum(bits_hat != bits) / bits.size)


# ── Flat <-> complex helpers (numpy) ──────────────────────────────────
# Flat layout used everywhere (dataset, models, loss, evaluation):
#   H_flat = [Re(H).ravel() | Im(H).ravel()]   with H (K, Nt), row-major
#   W_flat = [Re(W).ravel() | Im(W).ravel()]   with W (Nt, K), row-major

def complex_to_flat(X: np.ndarray) -> np.ndarray:
    """(B, ...) complex -> (B, 2*prod(...)) float32 in the [Re block | Im block] layout."""
    B = X.shape[0]
    return np.concatenate([X.real.reshape(B, -1), X.imag.reshape(B, -1)], axis=1).astype(np.float32)

def flat_to_complex_W(W_flat, Nt_, K_):
    half = Nt_ * K_
    return W_flat[:half].reshape(Nt_, K_) + 1j * W_flat[half:].reshape(Nt_, K_)

def flat_to_complex_H(H_flat, K_, Nt_):
    half = K_ * Nt_
    return H_flat[:half].reshape(K_, Nt_) + 1j * H_flat[half:].reshape(K_, Nt_)


# ── WMMSE targets for supervised pre-training ─────────────────────────
# wmmse_bf() is an iterative per-sample numpy routine with no cheap batched/GPU
# form, so labels are generated once, in a process pool, and cached to disk
# with the rest of the dataset.

def _wmmse_flat_single(args):
    H_hat_k, snr, n_iter, tol = args
    W = wmmse_bf(H_hat_k, snr, P_total=1.0, n_iter=n_iter, tol=tol)   # (Nt, K) complex
    return np.concatenate([W.real.ravel(), W.imag.ravel()]).astype(np.float32)

def wmmse_targets_batch(H_hat_complex: np.ndarray, snr_db: np.ndarray,
                        n_iter: int = 30, tol: float = 1e-4,
                        n_workers: int = None) -> np.ndarray:
    """
    H_hat_complex : (B, K, Nt) complex
    snr_db        : (B,) float
    returns       : (B, 2*Nt*K) float32 — flattened WMMSE precoders, one per row.
    """
    n_workers = n_workers or max(1, (os.cpu_count() or 2) - 1)
    args = [(H_hat_complex[i], float(snr_db[i]), n_iter, tol) for i in range(len(snr_db))]
    if n_workers == 1 or len(args) < 256:
        return np.stack([_wmmse_flat_single(a) for a in args], axis=0)
    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        out = list(ex.map(_wmmse_flat_single, args, chunksize=64))
    return np.stack(out, axis=0)

In [ ]:
# Import (or re-import after editing) the module written by the cell above
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
import bt_classical
importlib.reload(bt_classical)
from bt_classical import (power_normalise, mrt, zf_bf, mmse_bf, wmmse_bf,
                          compute_sinr, compute_sum_rate, compute_ber,
                          complex_to_flat, flat_to_complex_W, flat_to_complex_H,
                          wmmse_targets_batch)
print(f'[bt_classical] loaded from {bt_classical.__file__}')

## 5 — Dataset (chunked, low-RAM, Drive-cached)

* Training set: `ChunkedMIMODataset` — generated in chunks stored as `.npz` on disk; only
  one chunk is in RAM at a time (`ChunkSequentialSampler` never crosses chunk boundaries).
* Val / test: in-memory `MIMODataset`.
* Each sample is `(Ĥ_flat, H_flat, snr_db, W_wmmse_flat)`; `W_wmmse` is computed from `Ĥ`
  (the model never sees perfect CSI, so neither does its supervised target).

> **Layout fix vs. the original notebooks.** The originals stored channels as
> `concat([Re, Im], axis=-1).reshape(B, -1)`, i.e. `[Re(h₀) | Im(h₀) | Re(h₁) | …]`,
> while the models, the loss and the evaluation code all read `[Re(H) | Im(H)]`. Training
> and validation therefore ran on scrambled channels (e.g. "user 0" = `Re(h₀) + j·Re(h₄)`),
> and the WMMSE labels did not correspond to the channel the model saw. Here every
> stored array uses `complex_to_flat` (`[Re block | Im block]`), and `DATA_LAYOUT` is part
> of the cache hash so an old, scrambled cache can never be picked up by mistake.

In [ ]:
DATA_LAYOUT = 'reim_blocks_v2'   # bump if the stored array layout ever changes


def _dataset_config_hash(cfg: dict) -> str:
    s = _json.dumps(cfg, sort_keys=True, default=str)
    return hashlib.md5(s.encode()).hexdigest()[:10]


class MIMODataset(Dataset):
    """In-memory dataset of (H_hat, H, snr_db, W_wmmse) tuples (flat float32)."""
    def __init__(self, H_hat: np.ndarray, H: np.ndarray, snr_db: np.ndarray,
                 W_wmmse: np.ndarray):
        super().__init__()
        _, K_, Nt_ = H.shape
        self.H_hat   = torch.from_numpy(complex_to_flat(H_hat))
        self.H       = torch.from_numpy(complex_to_flat(H))
        self.snr     = torch.from_numpy(snr_db.astype(np.float32))
        self.W_wmmse = torch.from_numpy(W_wmmse.astype(np.float32))
        self.K, self.Nt = K_, Nt_

    def __len__(self):  return len(self.snr)
    def __getitem__(self, idx):
        return self.H_hat[idx], self.H[idx], self.snr[idx], self.W_wmmse[idx]

    def save_npz(self, path):
        np.savez(path, H_hat=self.H_hat.numpy(), H=self.H.numpy(),
                 snr=self.snr.numpy(), W_wmmse=self.W_wmmse.numpy(),
                 K=self.K, Nt=self.Nt)

    @classmethod
    def load_npz(cls, path):
        d = np.load(path)
        obj = cls.__new__(cls)
        obj.H_hat   = torch.from_numpy(d['H_hat'])
        obj.H       = torch.from_numpy(d['H'])
        obj.snr     = torch.from_numpy(d['snr'])
        obj.W_wmmse = torch.from_numpy(d['W_wmmse'])
        obj.K, obj.Nt = int(d['K']), int(d['Nt'])
        return obj


def _draw_samples(channel, n, rng, snr_range, pilot_snr_range):
    """One batch of (H, H_hat, snr_db) with per-sample data / pilot SNR."""
    H         = channel.generate(n, rng=rng)
    snr_db    = rng.uniform(*snr_range,       size=n).astype(np.float32)
    pilot_snr = rng.uniform(*pilot_snr_range, size=n)
    sigma_e   = np.sqrt(1.0 / (1.0 + 10 ** (pilot_snr / 10)))[:, None, None]
    noise     = sigma_e * (rng.standard_normal(H.shape) +
                           1j * rng.standard_normal(H.shape)) / np.sqrt(2)
    return H, H + noise, snr_db


class ChunkedMIMODataset(Dataset):
    """
    Generates the training set in chunks stored as .npz files (H_hat, H, snr,
    W_wmmse). Only one chunk is loaded into RAM at a time.

    If `cache_dir` already holds a complete, matching set of chunks + manifest,
    generation (including the expensive per-sample WMMSE solve) is skipped.
    If `cache_dir` is None, a throwaway local temp directory is used.
    """
    def __init__(self, n_samples, channel, chunk_size=10000,
                 snr_range=(0, 30), pilot_snr_range=(0, 20), seed=0,
                 wmmse_n_iter=30, wmmse_tol=1e-4,
                 cache_dir=None, force_regenerate=False, allow_generate=True):
        super().__init__()
        self.n_samples  = n_samples
        self.chunk_size = chunk_size
        self.K, self.Nt = channel.K, channel.Nt
        self._persistent = cache_dir is not None

        if cache_dir is not None:
            os.makedirs(cache_dir, exist_ok=True)
            self._cache_dir = cache_dir
        else:
            self._cache_dir = tempfile.mkdtemp(prefix='mimo_ds_')

        manifest_path = os.path.join(self._cache_dir, 'manifest.json')
        expected_config = {
            'n_samples': n_samples, 'chunk_size': chunk_size,
            'K': self.K, 'Nt': self.Nt,
            'snr_range': list(snr_range), 'pilot_snr_range': list(pilot_snr_range),
            'seed': seed, 'wmmse_n_iter': wmmse_n_iter, 'wmmse_tol': wmmse_tol,
            'layout': DATA_LAYOUT,
        }

        cached_ok = (not force_regenerate) and os.path.exists(manifest_path)
        saved_manifest, cached_files = None, []
        if cached_ok:
            with open(manifest_path, encoding='utf-8') as f:
                saved_manifest = _json.load(f)
            # Chunk paths are resolved relative to THIS cache dir (by file name), so a
            # cache that was moved, copied or unzipped elsewhere is still found.
            cached_files = [os.path.join(self._cache_dir, re.split(r'[\\/]', p)[-1])
                            for p in saved_manifest.get('chunk_files', [])]
            cached_ok = (saved_manifest.get('config') == expected_config and
                         len(cached_files) > 0 and all(os.path.exists(p) for p in cached_files))

        if cached_ok:
            print(f'[Dataset] Reusing cached chunks at {self._cache_dir}')
            self._chunk_files = cached_files
            self._chunk_sizes = saved_manifest['chunk_sizes']
        elif not allow_generate:
            raise RuntimeError(
                f'No complete, matching training-set cache in {self._cache_dir} and dataset '
                f'generation is disabled (a dataset zip was given). Expected config: {expected_config}')
        else:
            self._chunk_files = []
            self._chunk_sizes = []

            rng = np.random.default_rng(seed)
            n_chunks = int(np.ceil(n_samples / chunk_size))
            written  = 0

            print(f'  Writing {n_chunks} chunks to {self._cache_dir} ...', end=' ', flush=True)
            for c in range(n_chunks):
                size = min(chunk_size, n_samples - written)
                H, H_hat, snr_db = _draw_samples(channel, size, rng, snr_range, pilot_snr_range)
                W_wmmse = wmmse_targets_batch(H_hat, snr_db,
                                              n_iter=wmmse_n_iter, tol=wmmse_tol)
                path = os.path.join(self._cache_dir, f'chunk_{c:04d}.npz')
                np.savez(path, H_hat=complex_to_flat(H_hat), H=complex_to_flat(H),
                         snr=snr_db, W_wmmse=W_wmmse)
                self._chunk_files.append(path)
                self._chunk_sizes.append(size)
                written += size
                print(f'{c+1}', end=' ', flush=True)
            print('done.')

            with open(manifest_path, 'w', encoding='utf-8') as f:
                _json.dump({'config': expected_config,
                            'chunk_files': self._chunk_files,
                            'chunk_sizes': self._chunk_sizes}, f)

        # Flat index: each entry is (chunk_idx, row_within_chunk)
        self._index_map = [(c, i) for c, sz in enumerate(self._chunk_sizes) for i in range(sz)]
        self._loaded_chunk_idx  = None
        self._loaded_chunk_data = None

    def _load_chunk(self, chunk_idx):
        if self._loaded_chunk_idx != chunk_idx:
            data = np.load(self._chunk_files[chunk_idx])
            self._loaded_chunk_data = (
                torch.from_numpy(data['H_hat']),
                torch.from_numpy(data['H']),
                torch.from_numpy(data['snr']),
                torch.from_numpy(data['W_wmmse']),
            )
            self._loaded_chunk_idx = chunk_idx

    def __len__(self): return self.n_samples

    def __getitem__(self, idx):
        chunk_idx, row = self._index_map[idx]
        self._load_chunk(chunk_idx)
        H_hat, H, snr, W_wmmse = self._loaded_chunk_data
        return H_hat[row], H[row], snr[row], W_wmmse[row]

    def cleanup(self):
        """Deletes the chunk cache only if it is a local throwaway temp dir."""
        if self._persistent:
            print(f'[Dataset] Cache is persistent at {self._cache_dir} — not deleting.')
            return
        shutil.rmtree(self._cache_dir, ignore_errors=True)


class ChunkSequentialSampler(torch.utils.data.Sampler):
    """Yields indices chunk-by-chunk (chunk order and within-chunk order shuffled
    each epoch), never crossing chunk boundaries, so only one chunk is RAM-resident."""
    def __init__(self, dataset, shuffle=True, seed=0):
        self.dataset = dataset
        self.shuffle = shuffle
        self.seed    = seed
        self.epoch   = 0

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return len(self.dataset)

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        chunk_sizes = self.dataset._chunk_sizes
        chunk_order = rng.permutation(len(chunk_sizes)).tolist()
        offsets = np.concatenate([[0], np.cumsum(chunk_sizes)[:-1]]).astype(int)
        indices = []
        for c in chunk_order:
            chunk_indices = np.arange(offsets[c], offsets[c] + chunk_sizes[c])
            if self.shuffle:
                rng.shuffle(chunk_indices)
            indices.extend(chunk_indices.tolist())
        return iter(indices)


def generate_dataset(n_samples, channel, snr_range=(0, 30),
                     pilot_snr_range=(0, 20), seed=42,
                     wmmse_n_iter=30, wmmse_tol=1e-4,
                     cache_path=None, force_regenerate=False, allow_generate=True):
    """In-memory MIMODataset (val/test); loads from cache_path if it exists."""
    if cache_path is not None and os.path.exists(cache_path) and not force_regenerate:
        print(f'[Dataset] Loading cached dataset from {cache_path}')
        return MIMODataset.load_npz(cache_path)
    if not allow_generate:
        raise RuntimeError(f'Cached dataset {cache_path} not found and dataset generation '
                           f'is disabled (a dataset zip was given).')

    rng = np.random.default_rng(seed)
    H, H_hat, snr_db = _draw_samples(channel, n_samples, rng, snr_range, pilot_snr_range)
    W_wmmse = wmmse_targets_batch(H_hat, snr_db, n_iter=wmmse_n_iter, tol=wmmse_tol)
    ds = MIMODataset(H_hat, H, snr_db, W_wmmse)

    if cache_path is not None:
        os.makedirs(os.path.dirname(cache_path), exist_ok=True)
        ds.save_npz(cache_path)
        print(f'[Dataset] Saved to {cache_path}')
    return ds


def dataset_cache_names(Nt, K, L, array_type, n_train, n_val, n_test, snr_range,
                        pilot_snr_range, chunk_size, wmmse_n_iter, wmmse_tol):
    """Cache-folder names for a dataset config: (current 'cfg_<hash>', legacy 'cfg_<hash>').
    The legacy name is what the two ORIGINAL notebooks used (same config, no layout tag,
    channels stored in the old per-user interleaved layout)."""
    cfg = {
        'Nt': Nt, 'K': K, 'L': L, 'array_type': array_type,
        'n_train': n_train, 'n_val': n_val, 'n_test': n_test,
        'snr_range': list(snr_range), 'pilot_snr_range': list(pilot_snr_range),
        'chunk_size': chunk_size,
        'wmmse_n_iter': wmmse_n_iter, 'wmmse_tol': wmmse_tol,
    }
    return (f'cfg_{_dataset_config_hash({**cfg, "layout": DATA_LAYOUT})}',
            f'cfg_{_dataset_config_hash(cfg)}')


def get_dataloaders(Nt, K, L, n_train, n_val, n_test, batch_size,
                    chunk_size=10000, array_type='ULA',
                    snr_range=(0, 30), pilot_snr_range=(0, 20),
                    wmmse_n_iter=30, wmmse_tol=1e-4,
                    cache_dir=None, force_regenerate=False, allow_generate=True):
    """
    Build train / val / test DataLoaders. With `cache_dir`, the whole dataset
    (train chunks + manifest + val/test .npz, incl. WMMSE labels) is cached in a
    config-hashed subfolder and reused automatically by later runs.
    allow_generate=False raises instead of generating anything that isn't cached.

    Returns (train_loader, val_loader, test_loader, channel, train_ds, train_sampler).
    """
    channel = MassiveMIMOChannel(Nt=Nt, K=K, L=L, array_type=array_type,
                                 Nv=4, Nh=max(1, Nt // 4), ang_spread=15.0)

    run_cache_dir = None
    if cache_dir is not None:
        cfg_name, _ = dataset_cache_names(Nt, K, L, array_type, n_train, n_val, n_test,
                                          snr_range, pilot_snr_range, chunk_size,
                                          wmmse_n_iter, wmmse_tol)
        run_cache_dir = os.path.join(cache_dir, cfg_name)
        os.makedirs(run_cache_dir, exist_ok=True)
        print(f'[Dataset] Cache dir: {run_cache_dir}')

    train_cache = os.path.join(run_cache_dir, 'train')    if run_cache_dir else None
    val_cache   = os.path.join(run_cache_dir, 'val.npz')  if run_cache_dir else None
    test_cache  = os.path.join(run_cache_dir, 'test.npz') if run_cache_dir else None
    common = dict(snr_range=snr_range, pilot_snr_range=pilot_snr_range,
                  wmmse_n_iter=wmmse_n_iter, wmmse_tol=wmmse_tol,
                  force_regenerate=force_regenerate, allow_generate=allow_generate)

    print(f'[Dataset] Train: {n_train} samples, chunk_size={chunk_size} ...')
    train_ds = ChunkedMIMODataset(n_train, channel, chunk_size=chunk_size, seed=0,
                                  cache_dir=train_cache, **common)
    train_sampler = ChunkSequentialSampler(train_ds, shuffle=True, seed=0)

    print(f'[Dataset] Val: {n_val} samples ...')
    val_ds  = generate_dataset(n_val,  channel, seed=1, cache_path=val_cache,  **common)
    print(f'[Dataset] Test: {n_test} samples ...')
    test_ds = generate_dataset(n_test, channel, seed=2, cache_path=test_cache, **common)

    kw = dict(num_workers=0, pin_memory=torch.cuda.is_available())
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=train_sampler, **kw)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, **kw)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, **kw)
    print(f'[Dataset] Done. Input dim = {2 * K * Nt}')
    return train_loader, val_loader, test_loader, channel, train_ds, train_sampler


# ── Using a dataset_cache uploaded as a zip (e.g. downloaded from Google Drive) ──

def _legacy_to_blocks(A, K, Nt):
    """Old per-user interleaved layout [Re(h0) Im(h0) Re(h1) ...] -> [Re block | Im block]."""
    B = A.shape[0]
    X = A.reshape(B, K, 2 * Nt)
    return np.concatenate([X[:, :, :Nt].reshape(B, -1),
                           X[:, :, Nt:].reshape(B, -1)], axis=1).astype(np.float32)


def convert_legacy_cache(src_dir, dst_dir, K, Nt):
    """Convert a cache written by the ORIGINAL notebooks into the current layout.
    Only H_hat and H are re-arranged (a pure permutation of stored numbers); snr and
    the WMMSE labels were already stored correctly, so nothing is re-solved.
    The result is identical to what this notebook would generate from scratch."""
    os.makedirs(os.path.join(dst_dir, 'train'), exist_ok=True)
    with open(os.path.join(src_dir, 'train', 'manifest.json'), encoding='utf-8') as f:
        man = _json.load(f)
    files = []
    print(f'[Dataset] Converting legacy cache {src_dir} -> {dst_dir} ...', end=' ', flush=True)
    for i, p in enumerate(man['chunk_files']):
        name = re.split(r'[\\/]', p)[-1]
        d = np.load(os.path.join(src_dir, 'train', name))
        out = os.path.join(dst_dir, 'train', name)
        np.savez(out, H_hat=_legacy_to_blocks(d['H_hat'], K, Nt), H=_legacy_to_blocks(d['H'], K, Nt),
                 snr=d['snr'], W_wmmse=d['W_wmmse'])
        files.append(out)
        print(i + 1, end=' ', flush=True)
    with open(os.path.join(dst_dir, 'train', 'manifest.json'), 'w', encoding='utf-8') as f:
        _json.dump({'config': {**man['config'], 'layout': DATA_LAYOUT},
                    'chunk_files': files, 'chunk_sizes': man['chunk_sizes']}, f)
    for split in ('val', 'test'):
        d = np.load(os.path.join(src_dir, f'{split}.npz'))
        np.savez(os.path.join(dst_dir, f'{split}.npz'),
                 H_hat=_legacy_to_blocks(d['H_hat'], K, Nt), H=_legacy_to_blocks(d['H'], K, Nt),
                 snr=d['snr'], W_wmmse=d['W_wmmse'], K=d['K'], Nt=d['Nt'])
    print('done.')


def prepare_dataset_cache_from_zip(zip_pattern, extract_root, cfg_name, legacy_cfg_name, K, Nt):
    """
    Unzip every file matching `zip_pattern` (Drive splits big folders into
    ...-1-001.zip, ...-1-002.zip, ...) into `extract_root`, locate the dataset
    folder for this config and return the cache dir to pass to get_dataloaders().
      * current-layout folder (cfg_name) found -> used as is
      * only the ORIGINAL notebooks' folder (legacy_cfg_name) found -> converted once
      * nothing matching -> clear error listing what the zip contains
    """
    import glob, zipfile
    zips = sorted(glob.glob(zip_pattern))
    if not zips:
        raise FileNotFoundError(
            f'No file matches DATASET_ZIP = {zip_pattern!r}. Upload the dataset zip to /content '
            f'(Files panel) or fix DATASET_ZIP in the Configuration cell.')
    marker = os.path.join(extract_root, '.extracted_' + '_'.join(os.path.basename(z) for z in zips))
    if not os.path.exists(marker):
        os.makedirs(extract_root, exist_ok=True)
        for z in zips:
            print(f'[Dataset] Extracting {z} ...', flush=True)
            with zipfile.ZipFile(z) as zf:
                zf.extractall(extract_root)
        open(marker, 'w').close()

    found = {}
    for root, dirs, _ in os.walk(extract_root):
        for d in dirs:
            if d.startswith('cfg_'):
                found.setdefault(d, os.path.join(root, d))
    print(f'[Dataset] Dataset folders in zip: {sorted(found) or "none"}')

    if cfg_name in found:
        print(f'[Dataset] Using {found[cfg_name]}')
        return os.path.dirname(found[cfg_name])
    if legacy_cfg_name in found:
        print(f'[Dataset] Found {legacy_cfg_name} written by the original notebooks '
              f'(old channel layout) — converting it, no WMMSE re-solve needed.')
        conv_root = os.path.join(extract_root, 'converted')
        dst = os.path.join(conv_root, cfg_name)
        if not os.path.exists(os.path.join(dst, 'test.npz')):
            convert_legacy_cache(found[legacy_cfg_name], dst, K, Nt)
        return conv_root

    details = []
    for name, path in sorted(found.items()):
        mp = os.path.join(path, 'train', 'manifest.json')
        cfg = _json.load(open(mp, encoding='utf-8'))['config'] if os.path.exists(mp) else '?'
        details.append(f'  {name}: {cfg}')
    raise RuntimeError(
        f'The zip holds no dataset for this configuration (expected folder {cfg_name}, or '
        f'{legacy_cfg_name} from the original notebooks). Check Nt/K/L/ARRAY, N_TRAIN/N_VAL/'
        f'N_TEST, CHUNK_SIZE, SNR ranges and WMMSE settings. Found:\n' + '\n'.join(details))

## 6 — Models

**BeamTransformer (`BeamNet`, proposed).** Each user's channel (`Nt` complex → `2·Nt` real)
is one token; an SNR token is prepended; a Pre-LN Transformer encoder applies full
`(K+1)`-way self-attention at every layer; user tokens are projected back to precoder
columns and power-normalised.

```
H_hat_flat (B, 2·K·Nt) + snr_db (B,)
   ↓ embed each user's channel → (B, K, d_model)
   ↓ prepend SNR token         → (B, K+1, d_model)
   ↓ Pre-LN TransformerEncoder (n_layers × n_heads)
   ↓ project K user tokens     → (B, K, 2·Nt)
   ↓ rearrange + ‖W‖_F = 1     → W_flat (B, 2·Nt·K)
```

**Baselines**

| Name | Paper | Idea |
|---|---|---|
| BlackboxFNN (`XiaBNN`) | Xia et al. 2020 [B1] | CNN on the raw channel predicts (p, λ); W recovered in closed form |
| IAIDNN | Hu et al. 2021 [B2] | Deep-unfolded WMMSE with learnable approximations of the Nt×Nt inverse |
| FNN-Zhang (`ZhangFNN`) | Zhang et al. 2022 [B3] | As [B1] but the input is the K×K Gram matrix H Hᴴ |

**Architecture ablation (Transformer → MLP)**

| Name | Idea | Question it answers |
|---|---|---|
| BeamNet-UserMLP (A) | BeamNet with every self-attention sublayer removed; each user token goes through the same residual FFN stack independently; SNR embedding added to every token | Does cross-user attention (interference coordination) drive the gain, or is it per-user processing such as CSI denoising? |
| FlatMLP (B) | Plain MLP on the whole flattened channel + SNR → W (≈ parameter-matched to BeamNet) | Does the channel-token / Transformer design beat a generic network of the same size? |

All models share `forward(H_hat_flat, snr_db) -> W_flat` and the differentiable
negative-sum-rate loss.

In [ ]:
# ── Shared layers, loss, evaluation ───────────────────────────────────

class PowerNorm(nn.Module):
    """W <- sqrt(P_total) * W / ||W||_F  (per sample)."""
    def __init__(self, P_total: float = 1.0):
        super().__init__()
        self.P_total = P_total

    def forward(self, x):
        norm = torch.linalg.norm(x, dim=1, keepdim=True).clamp(min=1e-8)
        return x * (self.P_total ** 0.5) / norm


def sinr_sumrate_loss(W_flat, H_flat, snr_db, Nt, K):
    """Negative mean sum-rate (differentiable through the SINR)."""
    B = W_flat.shape[0]
    W = torch.complex(W_flat[:, :Nt*K].view(B, Nt, K), W_flat[:, Nt*K:].view(B, Nt, K))
    H = torch.complex(H_flat[:, :K*Nt].view(B, K, Nt), H_flat[:, K*Nt:].view(B, K, Nt))
    HW        = H @ W
    noise_var = 1.0 / (10.0 ** (snr_db / 10.0))
    sig   = torch.abs(torch.diagonal(HW, dim1=1, dim2=2)) ** 2
    total = (torch.abs(HW) ** 2).sum(dim=-1)
    sinr  = sig / (total - sig + noise_var.unsqueeze(1) + 1e-10)
    return -torch.log2(1.0 + sinr).sum(dim=1).mean()


@torch.no_grad()
def evaluate_model(model, loader, device):
    """Mean sum-rate (bps/Hz) over a loader (evaluated on the TRUE channel H)."""
    model.eval()
    total, n = 0.0, 0
    for H_hat, H, snr, _ in loader:
        H_hat, H, snr = H_hat.to(device), H.to(device), snr.to(device)
        loss = sinr_sumrate_loss(model(H_hat, snr), H, snr, model.Nt, model.K)
        total += (-loss.item()) * H_hat.shape[0]
        n     += H_hat.shape[0]
    model.train()
    return total / n


# ── Complex-channel helpers used by the baselines (torch) ─────────────

def flat_to_H(H_flat, K, Nt):
    B, half = H_flat.shape[0], K * Nt
    return torch.complex(H_flat[:, :half].view(B, K, Nt), H_flat[:, half:].view(B, K, Nt))

def W_to_flat(W):
    """W: (B, Nt, K) complex -> (B, 2*Nt*K) real."""
    B = W.shape[0]
    return torch.cat([W.real.reshape(B, -1), W.imag.reshape(B, -1)], dim=1)

def recover_W_from_pl(H, p, lam, sigma2, eps=1e-8):
    """
    Closed-form optimal MISO precoder structure (Xia et al. Eq. 24/28, Zhang et al. Eq. 8):
        w_k = sqrt(p_k) * normalize( (sigma^2 I + sum_j lam_j h_j h_j^H)^-1 h_k )
    H: (B,K,Nt) complex, row k = h_k^H.  p, lam: (B,K) real, each summing to P_total.
    Returns W: (B,Nt,K) complex, column k = w_k.
    """
    B, K, Nt = H.shape
    Hc = H.conj()
    weighted_Hc = Hc * lam.to(H.dtype).unsqueeze(-1)
    C = torch.einsum('bkn,bkm->bnm', weighted_Hc, H)       # sum_k lam_k h_k h_k^H
    eye = torch.eye(Nt, dtype=H.dtype, device=H.device).unsqueeze(0)
    C = C + sigma2.to(H.dtype).view(B, 1, 1) * eye
    Wdir = torch.linalg.solve(C, Hc.permute(0, 2, 1))      # (B,Nt,K)
    Wdir = Wdir / torch.linalg.norm(Wdir, dim=1, keepdim=True).clamp(min=eps)
    return Wdir * torch.sqrt(p.clamp(min=0)).to(H.dtype).unsqueeze(1)

def normalize_pl(raw, K, P_total=1.0, eps=1e-8):
    """Split a (B,2K) sigmoid output into (p, lam), each L1-normalised to P_total
    (the scaling layer, Xia Eq. 27)."""
    p_raw, lam_raw = raw[:, :K], raw[:, K:]
    p   = P_total * p_raw   / p_raw.sum(1, keepdim=True).clamp(min=eps)
    lam = P_total * lam_raw / lam_raw.sum(1, keepdim=True).clamp(min=eps)
    return p, lam


# ══════════════════════════════════════════════════════════════════════
# BeamTransformer (proposed)
# ══════════════════════════════════════════════════════════════════════
class BeamNet(nn.Module):
    """Channel-token Pre-LN Transformer beamformer with an SNR conditioning token."""
    def __init__(self, Nt=32, K=8, d_model=128, n_layers=5,
                 n_heads=4, dropout=0.1, P_total=1.0):
        super().__init__()
        self.Nt, self.K = Nt, K
        self.P_total, self.d_model, self.n_layers, self.n_heads = P_total, d_model, n_layers, n_heads
        token_in = 2 * Nt

        self.token_embed = nn.Linear(token_in, d_model)
        self.snr_embed   = nn.Linear(1, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, norm_first=True)   # Pre-LN
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers,
                                                 norm=nn.LayerNorm(d_model))
        self.output_proj = nn.Linear(d_model, token_in)
        self.power_norm  = PowerNorm(P_total)

    def forward(self, H_hat_flat: torch.Tensor, snr_db: torch.Tensor) -> torch.Tensor:
        B, K, Nt = H_hat_flat.shape[0], self.K, self.Nt
        half = K * Nt
        H_re = H_hat_flat[:, :half].view(B, K, Nt)
        H_im = H_hat_flat[:, half:].view(B, K, Nt)
        x       = self.token_embed(torch.cat([H_re, H_im], dim=-1))            # (B, K, d)
        snr_tok = self.snr_embed((snr_db / 30.0).unsqueeze(1)).unsqueeze(1)    # (B, 1, d)
        out     = self.transformer(torch.cat([snr_tok, x], dim=1))             # (B, K+1, d)
        w_tok   = self.output_proj(out[:, 1:, :])                              # (B, K, 2*Nt)
        W_re = w_tok[:, :, :Nt].permute(0, 2, 1).reshape(B, -1)
        W_im = w_tok[:, :, Nt:].permute(0, 2, 1).reshape(B, -1)
        return self.power_norm(torch.cat([W_re, W_im], dim=1))


# ══════════════════════════════════════════════════════════════════════
# BlackboxFNN == XiaBNN  [B1]
# ══════════════════════════════════════════════════════════════════════
class XiaBNN(nn.Module):
    """
    CNN predicts key features (p, lambda) in R^{2K}; W is recovered with the
    closed-form optimal-structure formula. Architecture per the paper's Table I:
    2 conv layers (3x3, pad 1) + BN + ReLU, flatten, FC -> 2K, sigmoid, on the
    I/Q-stacked channel as a (2, K*Nt) image.
    Disclosed deviation: a small SNR embedding is concatenated before the FC layer,
    since one model is trained across an SNR range.
    """
    def __init__(self, Nt=32, K=8, n_kernels=8, P_total=1.0):
        super().__init__()
        self.Nt, self.K, self.P_total = Nt, K, P_total
        self.conv = nn.Sequential(
            nn.Conv2d(1, n_kernels, 3, padding=1), nn.BatchNorm2d(n_kernels), nn.ReLU(True),
            nn.Conv2d(n_kernels, n_kernels, 3, padding=1), nn.BatchNorm2d(n_kernels), nn.ReLU(True),
        )
        self.snr_embed = nn.Linear(1, 32)
        self.fc = nn.Linear(n_kernels * 2 * Nt * K + 32, 2 * K)
        self.pnorm = PowerNorm(P_total)

    def forward(self, H_flat, snr_db):
        B = H_flat.shape[0]
        H = flat_to_H(H_flat, self.K, self.Nt)
        img  = torch.stack([H.real, H.imag], dim=1).reshape(B, 1, 2, self.K * self.Nt)
        feat = self.conv(img).reshape(B, -1)
        s    = self.snr_embed((snr_db / 30.0).unsqueeze(1))
        raw  = torch.sigmoid(self.fc(torch.cat([feat, s], dim=1)))
        p, lam = normalize_pl(raw, self.K, self.P_total)
        W = recover_W_from_pl(H, p, lam, 10.0 ** (-snr_db / 10.0))
        return self.pnorm(W_to_flat(W))


# ══════════════════════════════════════════════════════════════════════
# IAIDNN  [B2]
# ══════════════════════════════════════════════════════════════════════
class IAIDNNLayer(nn.Module):
    """
    One unfolded WMMSE iteration (paper Eqs. 15-16), MISO case (U_k, W_k scalars,
    updated exactly). The Nt x Nt inversion in the V-update is replaced by the
    paper's learnable approximation B^+ X_k + B Y_k + Z_k + O_k (B^+ = diagonal
    reciprocal), except in the last layer which uses the exact solve.
    Disclosed deviation: autograd instead of the paper's closed-form chain rule.
    """
    def __init__(self, Nt, K, last_layer=False):
        super().__init__()
        self.Nt, self.K, self.last_layer = Nt, K, last_layer
        if not last_layer:
            # X = I, Y = Z = O = 0 at init -> layer starts as the Jacobi-style B^+ @ rhs
            self.Xr = nn.Parameter(torch.eye(Nt).unsqueeze(0).repeat(K, 1, 1))
            self.Xi = nn.Parameter(torch.zeros(K, Nt, Nt))
            self.Yr = nn.Parameter(torch.zeros(K, Nt, Nt))
            self.Yi = nn.Parameter(torch.zeros(K, Nt, Nt))
            self.Zr = nn.Parameter(torch.zeros(K, Nt, Nt))
            self.Zi = nn.Parameter(torch.zeros(K, Nt, Nt))
            self.Or = nn.Parameter(torch.zeros(K, Nt))
            self.Oi = nn.Parameter(torch.zeros(K, Nt))

    def forward(self, V, H, sigma2, eps=1e-8):
        B, K, Nt = H.shape
        # ---- U, W updates (exact scalars) ----
        HV   = torch.einsum('bkn,bnm->bkm', H, V)
        HVkk = torch.diagonal(HV, dim1=1, dim2=2)
        Vpow = (V.abs() ** 2).sum(dim=(1, 2))
        A    = sigma2.unsqueeze(1) * Vpow.unsqueeze(1) + (HV.abs() ** 2).sum(dim=2)
        U    = HVkk / (A.to(H.dtype) + eps)
        E    = 1.0 - U.conj() * HVkk
        Wsc_r = (1.0 / (E + eps)).real.clamp(min=eps)

        # ---- B matrix (Nt x Nt) ----
        coef = (U.abs() ** 2) * Wsc_r
        diag_term = (sigma2 * coef.sum(dim=1)).to(H.dtype)
        Hc = H.conj()
        outer = torch.einsum('bkn,bkm->bnm', Hc * coef.to(H.dtype).unsqueeze(-1), H)
        eye = torch.eye(Nt, dtype=H.dtype, device=H.device).unsqueeze(0)
        Bmat = diag_term.view(-1, 1, 1) * eye + outer
        rhs  = (Hc * (U * Wsc_r.to(H.dtype)).unsqueeze(-1)).permute(0, 2, 1)   # (B, Nt, K)

        if self.last_layer:
            Vnew = torch.linalg.solve(Bmat, rhs)
        else:
            # Apply X, Y, Z, B, B^+ to the RHS vectors (O(Nt^2) each), batched over users.
            Bplus = torch.diag_embed(1.0 / (torch.diagonal(Bmat, dim1=1, dim2=2) + eps))
            X = torch.complex(self.Xr, self.Xi)
            Y = torch.complex(self.Yr, self.Yi)
            Z = torch.complex(self.Zr, self.Zi)
            O = torch.complex(self.Or, self.Oi)
            r   = rhs.permute(0, 2, 1)                                 # (B, K, Nt)
            Xr_ = torch.einsum('knm,bkm->bkn', X, r)
            Yr_ = torch.einsum('knm,bkm->bkn', Y, r)
            Zr_ = torch.einsum('knm,bkm->bkn', Z, r)
            t1  = torch.einsum('bnm,bkm->bkn', Bplus, Xr_)
            t2  = torch.einsum('bnm,bkm->bkn', Bmat, Yr_)
            Vnew = (t1 + t2 + Zr_ + O.unsqueeze(0)).permute(0, 2, 1)  # (B, Nt, K)

        return Vnew / Vnew.norm(dim=(1, 2), keepdim=True).clamp(min=eps)


class IAIDNN(nn.Module):
    """Deep-unfolded WMMSE with n_layers learnable iterations, ZF initialisation.
    Trained end-to-end on the sum-rate loss (no WMMSE pre-training, per the paper)."""
    def __init__(self, Nt=32, K=8, n_layers=5, P_total=1.0):
        super().__init__()
        self.Nt, self.K, self.P_total = Nt, K, P_total
        self.layers = nn.ModuleList(
            [IAIDNNLayer(Nt, K, last_layer=(i == n_layers - 1)) for i in range(n_layers)])
        self.pnorm = PowerNorm(P_total)

    def forward(self, H_flat, snr_db):
        H = flat_to_H(H_flat, self.K, self.Nt)
        sigma2 = 10.0 ** (-snr_db / 10.0)
        HHc = H.conj().transpose(-2, -1) @ H
        eye = torch.eye(self.Nt, dtype=HHc.dtype, device=HHc.device).unsqueeze(0)
        V = torch.linalg.solve(HHc + 1e-6 * eye, H.conj().transpose(-2, -1))   # ZF-style V^0
        V = V / V.norm(dim=(1, 2), keepdim=True).clamp(min=1e-8)
        for layer in self.layers:
            V = layer(V, H, sigma2)
        return self.pnorm(W_to_flat(V))


# ══════════════════════════════════════════════════════════════════════
# FNN-Zhang == ZhangFNN  [B3]
# ══════════════════════════════════════════════════════════════════════
class ZhangFNN(nn.Module):
    """
    (p, lambda) + closed-form recovery as XiaBNN, but the network sees the K x K
    Gram matrix G = H H^H (paper Section IV-A1), with a Conv+BN+LeakyReLU backbone.
    Disclosed deviations: SNR embedding; the paper's recovery-module compression /
    pruning (Section IV-B) is not implemented.
    """
    def __init__(self, Nt=32, K=8, n_kernels=16, P_total=1.0):
        super().__init__()
        self.Nt, self.K, self.P_total = Nt, K, P_total
        self.conv = nn.Sequential(
            nn.Conv2d(1, n_kernels, 3, padding=1), nn.BatchNorm2d(n_kernels), nn.LeakyReLU(0.1, True),
            nn.Conv2d(n_kernels, n_kernels, 3, padding=1), nn.BatchNorm2d(n_kernels), nn.LeakyReLU(0.1, True),
        )
        self.snr_embed = nn.Linear(1, 32)
        self.fc = nn.Linear(n_kernels * 2 * K * K + 32, 2 * K)
        self.pnorm = PowerNorm(P_total)

    def forward(self, H_flat, snr_db):
        B = H_flat.shape[0]
        H = flat_to_H(H_flat, self.K, self.Nt)
        G = H @ H.conj().transpose(-2, -1)
        img  = torch.stack([G.real, G.imag], dim=1).reshape(B, 1, 2 * self.K, self.K)
        feat = self.conv(img).reshape(B, -1)
        s    = self.snr_embed((snr_db / 30.0).unsqueeze(1))
        raw  = torch.sigmoid(self.fc(torch.cat([feat, s], dim=1)))
        p, lam = normalize_pl(raw, self.K, self.P_total)
        W = recover_W_from_pl(H, p, lam, 10.0 ** (-snr_db / 10.0))
        return self.pnorm(W_to_flat(W))


# ══════════════════════════════════════════════════════════════════════
# Architecture ablation A — BeamNet-UserMLP (attention removed)
# ══════════════════════════════════════════════════════════════════════
class UserMLPBlock(nn.Module):
    """Pre-LN residual feed-forward block, identical to the FFN sublayer of
    BeamNet's nn.TransformerEncoderLayer (Linear d->4d, ReLU, dropout, Linear 4d->d)."""
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.ReLU(), nn.Dropout(dropout),
                                nn.Linear(4 * d_model, d_model), nn.Dropout(dropout))

    def forward(self, x):
        return x + self.ff(self.norm(x))


class BeamNetUserMLP(nn.Module):
    """
    Ablation A: BeamNet with every self-attention sublayer removed. Same channel
    tokens, embedding, output projection and power normalisation; each user's
    token is processed by the same stack of residual FFN blocks with NO
    information exchange between users. Because the SNR token can no longer
    reach the users through attention, the SNR embedding is added to every
    user token instead. Tests whether cross-user attention (i.e. interference
    coordination) is what drives BeamNet's gain.
    """
    def __init__(self, Nt=32, K=8, d_model=128, n_layers=5, dropout=0.1, P_total=1.0):
        super().__init__()
        self.Nt, self.K, self.d_model, self.n_layers = Nt, K, d_model, n_layers
        self.token_embed = nn.Linear(2 * Nt, d_model)
        self.snr_embed   = nn.Linear(1, d_model)
        self.blocks      = nn.Sequential(*[UserMLPBlock(d_model, dropout) for _ in range(n_layers)])
        self.norm        = nn.LayerNorm(d_model)
        self.output_proj = nn.Linear(d_model, 2 * Nt)
        self.power_norm  = PowerNorm(P_total)

    def forward(self, H_hat_flat, snr_db):
        B, K, Nt = H_hat_flat.shape[0], self.K, self.Nt
        half = K * Nt
        H_re = H_hat_flat[:, :half].view(B, K, Nt)
        H_im = H_hat_flat[:, half:].view(B, K, Nt)
        x = self.token_embed(torch.cat([H_re, H_im], dim=-1))                  # (B, K, d)
        x = x + self.snr_embed((snr_db / 30.0).unsqueeze(1)).unsqueeze(1)      # SNR on every token
        w_tok = self.output_proj(self.norm(self.blocks(x)))                    # (B, K, 2*Nt)
        W_re = w_tok[:, :, :Nt].permute(0, 2, 1).reshape(B, -1)
        W_im = w_tok[:, :, Nt:].permute(0, 2, 1).reshape(B, -1)
        return self.power_norm(torch.cat([W_re, W_im], dim=1))


# ══════════════════════════════════════════════════════════════════════
# Architecture ablation B — FlatMLP (no tokens, no attention)
# ══════════════════════════════════════════════════════════════════════
class FlatMLP(nn.Module):
    """
    Ablation B: generic fully-connected network. Input = the whole flattened
    channel [Re(H_hat) | Im(H_hat)] (2*K*Nt) concatenated with the normalised
    SNR; hidden layers Linear -> LayerNorm -> ReLU -> Dropout; output = W_flat
    (2*Nt*K) in the same layout as BeamNet, then power normalisation.
    Tests whether the channel-token / Transformer design beats a plain MLP of
    comparable size. Unlike BeamNet it is not permutation-equivariant in the
    users and its size depends on K.
    """
    def __init__(self, Nt=32, K=8, hidden=512, n_layers=3, dropout=0.1, P_total=1.0):
        super().__init__()
        self.Nt, self.K = Nt, K
        layers, d_in = [], 2 * K * Nt + 1
        for _ in range(n_layers):
            layers += [nn.Linear(d_in, hidden), nn.LayerNorm(hidden), nn.ReLU(), nn.Dropout(dropout)]
            d_in = hidden
        layers.append(nn.Linear(d_in, 2 * Nt * K))
        self.net = nn.Sequential(*layers)
        self.power_norm = PowerNorm(P_total)

    def forward(self, H_hat_flat, snr_db):
        x = torch.cat([H_hat_flat, (snr_db / 30.0).unsqueeze(1)], dim=1)
        return self.power_norm(self.net(x))


# ── Model registry ────────────────────────────────────────────────────

MODEL_REFS = {'BeamNet':         'BeamTransformer (proposed)',
              'BeamNet-UserMLP': 'Ablation A: BeamNet w/o attention',
              'FlatMLP':         'Ablation B: flat MLP',
              'BlackboxFNN':     'Xia et al. 2020 [B1]',
              'IAIDNN':          'Hu et al. 2021 [B2]',
              'FNN-Zhang':       'Zhang et al. 2022 [B3]'}
ABLATION_MODELS = ['BeamNet-UserMLP', 'FlatMLP']

def build_model(name, Nt, K):
    if name == 'BeamNet':
        return BeamNet(Nt=Nt, K=K, d_model=D_MODEL, n_layers=N_LAYERS,
                       n_heads=N_HEADS, dropout=DROPOUT)
    if name == 'BeamNet-UserMLP':
        return BeamNetUserMLP(Nt=Nt, K=K, d_model=USERMLP_D_MODEL,
                              n_layers=USERMLP_LAYERS, dropout=DROPOUT)
    if name == 'FlatMLP':
        return FlatMLP(Nt=Nt, K=K, hidden=FLATMLP_HIDDEN,
                       n_layers=FLATMLP_LAYERS, dropout=DROPOUT)
    if name == 'BlackboxFNN':
        return XiaBNN(Nt=Nt, K=K, n_kernels=XIA_KERNELS)
    if name == 'IAIDNN':
        return IAIDNN(Nt=Nt, K=K, n_layers=IAIDNN_LAYERS)
    if name == 'FNN-Zhang':
        return ZhangFNN(Nt=Nt, K=K, n_kernels=ZHANG_KERNELS)
    raise ValueError(f'Unknown model {name!r}')

def model_params(model):
    return sum(p.numel() for p in model.parameters())

print(f'Model parameter counts (Nt={Nt}, K={K}):')
for _name in MODELS_TO_RUN:
    print(f'  {_name:<16}: {model_params(build_model(_name, Nt, K)):>10,}')

## 7 — Training infrastructure

Same protocol for every model:
1. **MSE pre-training** (`PRETRAIN_EPOCHS`, Adam lr 1e-3) against the offline WMMSE target
   shipped by the dataloader — skipped for models in `NO_PRETRAIN_MODELS` (IAIDNN).
2. **Sum-rate training** with the differentiable SINR loss — AdamW + ReduceLROnPlateau +
   gradient clipping + early stopping; the best-validation checkpoint is kept.

In [ ]:
def train_model(model, name, train_loader, val_loader, device,
                lr, max_epochs, patience, pretrain_epochs=0, grad_clip=1.0):
    model = model.to(device)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch = ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=5, min_lr=1e-6)
    mse_loss = nn.MSELoss()
    best_val, no_impr, history = -float('inf'), 0, []
    ckpt_path = os.path.join(CHECKPOINT_DIR, f'{name}_best.pt')
    print(f'\n[Train] {name}  ({model_params(model):,} params)')

    # ── Pre-training (MSE -> offline WMMSE target, 4th item of each batch) ──
    if pretrain_epochs > 0:
        print(f'  Pre-training {pretrain_epochs} epochs (MSE -> WMMSE target) ...')
        pre_opt = optim.Adam(model.parameters(), lr=1e-3)
        for ep in range(1, pretrain_epochs + 1):
            if hasattr(train_loader.sampler, 'set_epoch'):
                train_loader.sampler.set_epoch(ep)
            model.train(); tl, n = 0.0, 0
            for H_hat, _H, snr, W_wmmse in train_loader:
                H_hat, snr, W_wmmse = H_hat.to(device), snr.to(device), W_wmmse.to(device)
                pre_opt.zero_grad()
                loss = mse_loss(model(H_hat, snr), W_wmmse)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                pre_opt.step()
                tl += loss.item() * H_hat.shape[0]; n += H_hat.shape[0]
            print(f'    Pre-train {ep:>2}/{pretrain_epochs}  MSE = {tl / n:.6f}')
        print('  Pre-training done.')

    # ── Sum-rate training ─────────────────────────────────────────────
    print(f'  {"Ep":>4}  {"TrainLoss":>10}  {"ValRate":>9}  {"LR":>9}  {"t":>6}')
    print('  ' + '-' * 48)
    for ep in range(1, max_epochs + 1):
        if hasattr(train_loader.sampler, 'set_epoch'):
            train_loader.sampler.set_epoch(1000 + ep)
        model.train(); tl, n = 0.0, 0; t0 = time.time()
        for H_hat, H, snr, _ in train_loader:
            H_hat, H, snr = H_hat.to(device), H.to(device), snr.to(device)
            opt.zero_grad()
            loss = sinr_sumrate_loss(model(H_hat, snr), H, snr, model.Nt, model.K)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()
            tl += loss.item() * H_hat.shape[0]; n += H_hat.shape[0]
        tl /= n
        vr = evaluate_model(model, val_loader, device)
        sch.step(vr)
        lr_now = opt.param_groups[0]['lr']
        print(f'  {ep:>4}  {tl:>10.4f}  {vr:>9.4f}  {lr_now:>9.2e}  {time.time() - t0:>5.1f}s')
        history.append({'epoch': ep, 'train_loss': tl, 'val_rate': vr, 'lr': lr_now})

        if vr > best_val:
            best_val, no_impr = vr, 0
            torch.save({'name': name, 'epoch': ep, 'val_rate': vr,
                        'Nt': model.Nt, 'K': model.K,
                        'state_dict': model.state_dict()}, ckpt_path)
        else:
            no_impr += 1
            if no_impr >= patience:
                print(f'  [Early stop] no improvement for {patience} epochs.')
                break

    curve_path = os.path.join(RESULTS_DIR, f'{name}_training_curve.csv')
    with open(curve_path, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=['epoch', 'train_loss', 'val_rate', 'lr'])
        w.writeheader(); w.writerows(history)
    print(f'  Best val = {best_val:.4f} bps/Hz   (curve: {curve_path})')
    return history, best_val


def load_model(model, name, path=None, device='cpu'):
    """Load a checkpoint into `model` (from `path`, else this run's CHECKPOINT_DIR).
    Returns (model, val_rate_stored_in_checkpoint)."""
    p = path or os.path.join(CHECKPOINT_DIR, f'{name}_best.pt')
    if not os.path.exists(p):
        raise FileNotFoundError(f'No checkpoint for {name} at {p}')
    ck = torch.load(p, map_location='cpu')
    model.load_state_dict(ck['state_dict'])
    val = ck.get('val_rate', float('nan'))
    print(f'[Load] {name}  epoch={ck.get("epoch", "?")}  val={val:.4f}  from {p}')
    return model.to(device).eval(), val

## 8 — Build dataset

If `DATASET_ZIP` is set (default on Colab: `/content/dataset_cache-*.zip`), the uploaded
zip is extracted and the dataset is loaded from it — nothing is generated. A cache written
by the original notebooks is recognised and converted to the current channel layout
automatically (no WMMSE re-solve).

In [ ]:
_allow_generate = True
if DATASET_ZIP and USE_DATASET_CACHE:
    _cfg_name, _legacy_name = dataset_cache_names(
        Nt, K, L, ARRAY, N_TRAIN, N_VAL, N_TEST, SNR_RANGE, PILOT_SNR_RANGE,
        CHUNK_SIZE, WMMSE_N_ITER, WMMSE_TOL)
    DATASET_CACHE_DIR = prepare_dataset_cache_from_zip(
        DATASET_ZIP, DATASET_EXTRACT_DIR, _cfg_name, _legacy_name, K, Nt)
    _allow_generate = False

train_loader, val_loader, test_loader, channel, train_ds, train_sampler = get_dataloaders(
    Nt=Nt, K=K, L=L, array_type=ARRAY,
    n_train=N_TRAIN, n_val=N_VAL, n_test=N_TEST,
    batch_size=BATCH_SIZE, chunk_size=CHUNK_SIZE,
    snr_range=SNR_RANGE, pilot_snr_range=PILOT_SNR_RANGE,
    wmmse_n_iter=WMMSE_N_ITER, wmmse_tol=WMMSE_TOL,
    cache_dir=(DATASET_CACHE_DIR if USE_DATASET_CACHE else None),
    force_regenerate=FORCE_REGENERATE_DATA and _allow_generate,
    allow_generate=_allow_generate,
)

## 9 — Train (or reload) all models

Models with a path in `PRETRAINED_CHECKPOINTS` are loaded; all others are trained here.

In [ ]:
all_models, all_histories, all_val_rates, model_source = {}, {}, {}, {}

for name in MODELS_TO_RUN:
    model = build_model(name, Nt, K)

    if SKIP_TRAINING[name]:
        print(f'\n[Skip] {name} — reusing saved checkpoint (no training)')
        all_models[name], all_val_rates[name] = load_model(
            model, name, path=PRETRAINED_CHECKPOINTS[name], device=DEVICE)
        model_source[name] = PRETRAINED_CHECKPOINTS[name]
        # Pick up the training curve saved by the run that produced the checkpoint
        # (<run>/results/<name>_training_curve.csv), so plots/report still show it.
        _curve = os.path.join(os.path.dirname(os.path.dirname(PRETRAINED_CHECKPOINTS[name])),
                              'results', f'{name}_training_curve.csv')
        if os.path.exists(_curve):
            with open(_curve, encoding='utf-8') as f:
                all_histories[name] = [{k: (int(v) if k == 'epoch' else float(v)) for k, v in r.items()}
                                       for r in csv.DictReader(f)]
            print(f'  (training curve loaded from {_curve})')
        continue

    pre_ep = 0 if name in NO_PRETRAIN_MODELS else PRETRAIN_EPOCHS
    hist, best_val = train_model(model, name, train_loader, val_loader, DEVICE,
                                 lr=LR, max_epochs=MAX_EPOCHS, patience=PATIENCE,
                                 pretrain_epochs=pre_ep, grad_clip=GRAD_CLIP)
    all_histories[name] = hist
    all_models[name], all_val_rates[name] = load_model(model, name, device=DEVICE)
    model_source[name] = f'trained in Run_{RUN_ID} (pre-train epochs: {pre_ep})'

train_ds.cleanup()   # no-op for the persistent Drive cache
print('\n[Done] All models ready.')

## 10 — Validation / test sum-rates

In [ ]:
all_test_rates = {name: evaluate_model(m, test_loader, DEVICE) for name, m in all_models.items()}

print(f'\n{"Model":<16}  {"Params":>10}  {"Val (bps/Hz)":>13}  {"Test (bps/Hz)":>14}')
print('-' * 60)
for name, m in all_models.items():
    print(f'{name:<16}  {model_params(m):>10,}  {all_val_rates[name]:>13.4f}  {all_test_rates[name]:>14.4f}')

with open(os.path.join(RESULTS_DIR, 'model_summary.csv'), 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['Model', 'Params', 'Val_bpsHz', 'Test_bpsHz', 'Source'])
    for name, m in all_models.items():
        w.writerow([name, model_params(m), f'{all_val_rates[name]:.4f}',
                    f'{all_test_rates[name]:.4f}', model_source[name]])
print('\n[Data] model_summary.csv saved.')

save_run_params(PARAMS_PATH, extra_lines=[
    '', '=== Results (val / test mean sum-rate, bps/Hz) ===',
    *[f'  {n:<14}: {all_val_rates[n]:.4f} / {all_test_rates[n]:.4f}' for n in all_models],
])

## 11 — Evaluation suite

Every plot is saved as PNG in `results/`, and its data as `.npz` + `.csv` in
`results/plot_data/` (for MATLAB). All methods are evaluated on the **true** channel
`H`; precoders are computed from the noisy estimate `Ĥ` (except *Perfect CSI*, which is
MMSE-BF computed from `H`). DL models are only plotted where they actually produce an
output — no placeholder values.

| # | File | Content |
|---|------|---------|
| a | `sumrate_vs_snr.png` | Sum-rate vs. SNR — all DL + classical |
| b | `ber_vs_snr.png` | BER vs. SNR (QPSK, ZF receiver) |
| c | `beampatterns.png` | Spatial beampatterns (classical) |
| d | `ablation_antenna.png` | Sum-rate vs. antenna count |
| e | `csi_quality.png` | Sum-rate vs. pilot SNR |
| f | `complexity_and_curves.png` | Params vs. sum-rate, val curves of all models |
| g | `<model>_training_curve.png` | Per-model training curve |
| h | `architecture_ablation.csv` | BeamNet vs. BeamNet-UserMLP / FlatMLP table |

In [ ]:
# ── Plot style & helpers ──────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 12,
    'legend.fontsize': 9, 'lines.linewidth': 2.0,
    'lines.markersize': 5, 'grid.alpha': 0.35,
    'axes.spines.top': False, 'axes.spines.right': False,
})

CMAP = {
    'BeamNet':     '#1565c0',
    'BeamNet-UserMLP': '#4fc3f7',
    'FlatMLP':     '#6d4c41',
    'BlackboxFNN': '#e65100',
    'IAIDNN':      '#2e7d32',
    'FNN-Zhang':   '#6a1b9a',
    'MRT':         '#9e9e9e',
    'ZF-BF':       '#f9a825',
    'MMSE-BF':     '#c62828',
    'WMMSE':       '#ad1457',
    'Perfect CSI': '#00695c',
}
MMAP = {'BeamNet': 's', 'BeamNet-UserMLP': 'o', 'FlatMLP': 'h', 'BlackboxFNN': '^', 'IAIDNN': 'D', 'FNN-Zhang': 'P',
        'MRT': 'x', 'ZF-BF': '+', 'MMSE-BF': 'v', 'WMMSE': '<', 'Perfect CSI': '*'}
LSTY = {'Perfect CSI': '--'}
CLASSICAL = ['MRT', 'ZF-BF', 'MMSE-BF', 'WMMSE', 'Perfect CSI']


def save_plot_data(fname, **arrays):
    """Save plot data as results/plot_data/<fname>.npz (+ .csv when shapes allow)."""
    d = os.path.join(RESULTS_DIR, 'plot_data'); os.makedirs(d, exist_ok=True)
    np.savez(os.path.join(d, fname + '.npz'), **{k: np.asarray(v) for k, v in arrays.items()})
    try:
        keys = list(arrays.keys())
        vals = [np.asarray(v).flatten() for v in arrays.values()]
        nr = max(len(v) for v in vals)
        with open(os.path.join(d, fname + '.csv'), 'w', newline='', encoding='utf-8') as f:
            w = csv.writer(f); w.writerow(keys)
            for row in zip(*[v.tolist() + [''] * (nr - len(v)) for v in vals]):
                w.writerow(row)
    except Exception:
        pass   # non-uniform shapes -- the npz is sufficient
    print(f'  [Data] plot_data/{fname}.npz/.csv')


def finish_fig(fig, fname):
    path = os.path.join(RESULTS_DIR, fname)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f'[Plot] {path}')


def dl_precoders(models_dict, H_hat, snr_db, device):
    """Run every DL model on a batch of noisy channels H_hat (B,K,Nt) at one SNR.
    Returns {name: list of complex W (Nt,K)}; models whose Nt/K don't match are skipped."""
    B, K_, Nt_ = H_hat.shape
    H_in  = torch.from_numpy(complex_to_flat(H_hat)).to(device)
    snr_t = torch.full((B,), float(snr_db), dtype=torch.float32, device=device)
    out = {}
    for name, m in models_dict.items():
        if m.Nt != Nt_ or m.K != K_:
            continue
        m.eval()
        with torch.no_grad():
            W = m(H_in, snr_t).cpu().numpy()
        out[name] = [flat_to_complex_W(W[i], Nt_, K_) for i in range(B)]
    return out

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 11a. Sum-rate vs. SNR
# ══════════════════════════════════════════════════════════════════════
def eval_sumrate_vs_snr(models_dict, n_samples, snr_range=(-5, 35), n_points=17,
                        pilot_snr_db=15.0, device='cpu'):
    snr_dbs = np.linspace(*snr_range, n_points)
    rng     = np.random.default_rng(5)
    H_true  = make_channel().generate(n_samples, rng=rng)
    H_hat   = add_csi_noise(H_true, pilot_snr_db, rng=rng)
    results = {k: [] for k in CLASSICAL + list(models_dict)}
    for snr in snr_dbs:
        rates = {k: [] for k in results}
        for i in range(n_samples):
            H, Hh = H_true[i], H_hat[i]
            rates['MRT']        .append(compute_sum_rate(H, mrt(Hh),           snr))
            rates['ZF-BF']      .append(compute_sum_rate(H, zf_bf(Hh),         snr))
            rates['MMSE-BF']    .append(compute_sum_rate(H, mmse_bf(Hh, snr),  snr))
            rates['WMMSE']      .append(compute_sum_rate(H, wmmse_bf(Hh, snr), snr))
            rates['Perfect CSI'].append(compute_sum_rate(H, mmse_bf(H, snr),   snr))
        for name, Ws in dl_precoders(models_dict, H_hat, snr, device).items():
            rates[name] = [compute_sum_rate(H_true[i], Ws[i], snr) for i in range(n_samples)]
        for k in results:
            results[k].append(np.mean(rates[k]))
    return snr_dbs, results

print('[Eval] Sum-rate vs SNR ...')
snr_dbs, sr_results = eval_sumrate_vs_snr(all_models, N_EVAL_SUMRATE,
                                          pilot_snr_db=EVAL_PILOT_SNR, device=DEVICE)
idx20 = int(np.argmin(np.abs(snr_dbs - EVAL_REF_SNR)))

fig, ax = plt.subplots(figsize=(8, 5))
for label in CLASSICAL:
    ax.plot(snr_dbs, sr_results[label], color=CMAP[label], marker=MMAP[label], markevery=3,
            linestyle=LSTY.get(label, '-'), alpha=0.7, label=label)
for label in all_models:
    ax.plot(snr_dbs, sr_results[label], color=CMAP[label], marker=MMAP[label], markevery=3,
            linewidth=2.2, label=label)
ax.set_xlabel('SNR (dB)'); ax.set_ylabel('Sum-Rate (bps/Hz)')
ax.set_title(f'Sum-Rate vs. SNR  —  Nt={Nt}, K={K}, pilot SNR={EVAL_PILOT_SNR:g} dB')
ax.legend(ncol=2); ax.grid(True)
finish_fig(fig, 'sumrate_vs_snr.png')
save_plot_data('sumrate_vs_snr', snr_dbs=snr_dbs, **{k: np.array(v) for k, v in sr_results.items()})

print(f'\nSum-rate @ {snr_dbs[idx20]:g} dB SNR (bps/Hz):')
_mmse_ref = sr_results['MMSE-BF'][idx20]
for k, v in sr_results.items():
    print(f'  {k:<12}: {v[idx20]:8.4f}   ({(v[idx20] - _mmse_ref) / _mmse_ref * 100:+6.1f}% vs MMSE-BF)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 11b. BER vs. SNR  (QPSK, ZF receiver)
# ══════════════════════════════════════════════════════════════════════
def eval_ber_vs_snr(models_dict, n_channels, snr_range=(-5, 30), n_points=15,
                    pilot_snr_db=15.0, n_symbols=5000, device='cpu'):
    snr_dbs = np.linspace(*snr_range, n_points)
    rng     = np.random.default_rng(7)
    H_true  = make_channel().generate(n_channels, rng=rng)
    H_hat   = add_csi_noise(H_true, pilot_snr_db, rng=rng)
    labels  = ['MRT', 'ZF-BF', 'MMSE-BF', 'WMMSE'] + list(models_dict)
    results = {k: [] for k in labels}
    for snr in snr_dbs:
        bers = {k: [] for k in labels}
        for i in range(n_channels):
            H, Hh = H_true[i], H_hat[i]
            bers['MRT']    .append(compute_ber(H, mrt(Hh),           snr, n_symbols, rng))
            bers['ZF-BF']  .append(compute_ber(H, zf_bf(Hh),         snr, n_symbols, rng))
            bers['MMSE-BF'].append(compute_ber(H, mmse_bf(Hh, snr),  snr, n_symbols, rng))
            bers['WMMSE']  .append(compute_ber(H, wmmse_bf(Hh, snr), snr, n_symbols, rng))
        for name, Ws in dl_precoders(models_dict, H_hat, snr, device).items():
            bers[name] = [compute_ber(H_true[i], Ws[i], snr, n_symbols, rng) for i in range(n_channels)]
        for k in labels:
            results[k].append(np.mean(bers[k]))
    return snr_dbs, results

print('[Eval] BER vs SNR ...')
snr_dbs_ber, ber_results = eval_ber_vs_snr(all_models, N_EVAL_BER, pilot_snr_db=EVAL_PILOT_SNR,
                                           n_symbols=N_BER_SYMBOLS, device=DEVICE)

fig, ax = plt.subplots(figsize=(8, 5))
for label in ['MRT', 'ZF-BF', 'MMSE-BF', 'WMMSE']:
    ax.semilogy(snr_dbs_ber, np.maximum(ber_results[label], 1e-5), color=CMAP[label],
                marker=MMAP[label], markevery=3, alpha=0.7, label=label)
for label in all_models:
    ax.semilogy(snr_dbs_ber, np.maximum(ber_results[label], 1e-5), color=CMAP[label],
                marker=MMAP[label], markevery=3, linewidth=2.2, label=label)
ax.set_xlabel('SNR (dB)'); ax.set_ylabel('BER')
ax.set_title(f'BER vs. SNR  (QPSK, ZF rx)  —  Nt={Nt}, K={K}')
ax.legend(ncol=2); ax.grid(True, which='both')
finish_fig(fig, 'ber_vs_snr.png')
save_plot_data('ber_vs_snr', snr_dbs=snr_dbs_ber, **{k: np.array(v) for k, v in ber_results.items()})

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 11c. Spatial beampatterns (classical beamformers, users on a ULA grid)
# ══════════════════════════════════════════════════════════════════════
def plot_beampatterns(Nt_, K_, fname='beampatterns.png'):
    angles     = np.deg2rad(np.linspace(-60, 60, K_))
    H          = ula_steering(angles, Nt_).astype(complex)
    theta_scan = np.linspace(-np.pi / 2, np.pi / 2, 500)
    a_scan     = ula_steering(theta_scan, Nt_)

    bfs = {'MRT': mrt(H), 'ZF-BF': zf_bf(H), 'MMSE-BF': mmse_bf(H, snr_db=20)}
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
    gain_data = {}
    for ax, (name, W) in zip(axes, bfs.items()):
        gains_k = []
        for k in range(K_):
            pdb = 20 * np.log10(np.abs(a_scan @ W[:, k]) + 1e-12)
            pdb_norm = pdb - pdb.max()
            ax.plot(np.rad2deg(theta_scan), pdb_norm, color=CMAP[name], alpha=0.75)
            gains_k.append(pdb_norm)
        gain_data[name] = np.array(gains_k)   # (K, 500)
        for ang in np.rad2deg(angles):
            ax.axvline(ang, color='red', linestyle=':', linewidth=1.0, alpha=0.6)
        ax.set_title(name, color=CMAP[name])
        ax.set_xlabel('Angle (degrees)'); ax.set_ylim(-50, 5); ax.grid(True, alpha=0.35)
    axes[0].set_ylabel('Normalised Gain (dB)')
    angle_str = ', '.join(f'{int(round(np.rad2deg(a)))}°' for a in angles)
    fig.suptitle(f'Spatial Beampatterns  —  K={K_} users at {angle_str}  |  Nt={Nt_} ULA',
                 fontsize=11, y=1.02)
    finish_fig(fig, fname)
    save_plot_data('beampatterns', theta_scan_deg=np.rad2deg(theta_scan),
                   user_angles_deg=np.rad2deg(angles), **gain_data)

print('[Eval] Beampatterns ...')
plot_beampatterns(Nt, K)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 11d. Ablation — antenna count
#      DL models are trained for one Nt, so they only appear at that Nt.
# ══════════════════════════════════════════════════════════════════════
def eval_ablation_antenna(models_dict, ant_counts=(8, 16, 32, 64), n_samples=600,
                          pilot_snr_db=15.0, snr_db=20.0, device='cpu'):
    labels  = ['MRT', 'ZF-BF', 'MMSE-BF'] + list(models_dict) + ['Perfect CSI']
    results = {k: [] for k in labels}
    for Nt_ in ant_counts:
        rng    = np.random.default_rng(10)
        H_true = make_channel(Nt_=Nt_).generate(n_samples, rng=rng)
        H_hat  = add_csi_noise(H_true, pilot_snr_db, rng=rng)
        rates  = {k: [] for k in labels}
        for i in range(n_samples):
            H, Hh = H_true[i], H_hat[i]
            rates['MRT']        .append(compute_sum_rate(H, mrt(Hh),             snr_db))
            rates['ZF-BF']      .append(compute_sum_rate(H, zf_bf(Hh),           snr_db))
            rates['MMSE-BF']    .append(compute_sum_rate(H, mmse_bf(Hh, snr_db), snr_db))
            rates['Perfect CSI'].append(compute_sum_rate(H, mmse_bf(H, snr_db),  snr_db))
        for name, Ws in dl_precoders(models_dict, H_hat, snr_db, device).items():
            rates[name] = [compute_sum_rate(H_true[i], Ws[i], snr_db) for i in range(n_samples)]
        for k in labels:
            results[k].append(np.mean(rates[k]) if rates[k] else np.nan)
    return list(ant_counts), results

print('[Eval] Ablation — antenna count ...')
ant_counts, ant_results = eval_ablation_antenna(all_models, n_samples=N_EVAL_ABLATION,
                                                pilot_snr_db=EVAL_PILOT_SNR,
                                                snr_db=EVAL_REF_SNR, device=DEVICE)

fig, ax = plt.subplots(figsize=(9, 4.8))
labels = list(ant_results)
x, width = np.arange(len(ant_counts)), 0.8 / len(labels)
for i, label in enumerate(labels):
    ax.bar(x + i * width, ant_results[label], width, label=label,
           color=CMAP[label], alpha=0.85, edgecolor='white')
ax.set_xticks(x + width * (len(labels) - 1) / 2)
ax.set_xticklabels([f'Nₜ={n}' for n in ant_counts])
ax.set_ylabel('Sum-Rate (bps/Hz)')
ax.set_title(f'Ablation: Antenna Count  —  K={K}, SNR={EVAL_REF_SNR:g} dB '
             f'(DL models trained for Nₜ={Nt} only)')
ax.legend(loc='upper left', ncol=2); ax.grid(axis='y')
finish_fig(fig, 'ablation_antenna.png')
save_plot_data('ablation_antenna', antenna_counts=np.array(ant_counts),
               **{k: np.array(v) for k, v in ant_results.items()})

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 11e. Ablation — CSI quality (sum-rate vs. pilot SNR)
# ══════════════════════════════════════════════════════════════════════
def eval_csi_quality(models_dict, n_samples=600, data_snr=20.0, device='cpu'):
    pilot_snrs = np.linspace(0, 20, 11)
    rng     = np.random.default_rng(11)
    H_true  = make_channel().generate(n_samples, rng=rng)
    labels  = ['ZF-BF', 'MMSE-BF', 'WMMSE'] + list(models_dict)
    results = {k: [] for k in labels}
    for p_snr in pilot_snrs:
        H_hat = add_csi_noise(H_true, p_snr, rng=rng)
        rates = {k: [] for k in labels}
        for i in range(n_samples):
            H, Hh = H_true[i], H_hat[i]
            rates['ZF-BF']  .append(compute_sum_rate(H, zf_bf(Hh),                data_snr))
            rates['MMSE-BF'].append(compute_sum_rate(H, mmse_bf(Hh, data_snr),    data_snr))
            rates['WMMSE']  .append(compute_sum_rate(H, wmmse_bf(Hh, data_snr),   data_snr))
        for name, Ws in dl_precoders(models_dict, H_hat, data_snr, device).items():
            rates[name] = [compute_sum_rate(H_true[i], Ws[i], data_snr) for i in range(n_samples)]
        for k in labels:
            results[k].append(np.mean(rates[k]))
    return pilot_snrs, results

print('[Eval] Ablation — CSI quality ...')
pilot_snrs, csi_results = eval_csi_quality(all_models, n_samples=N_EVAL_ABLATION,
                                           data_snr=EVAL_REF_SNR, device=DEVICE)

fig, ax = plt.subplots(figsize=(8, 5))
for label in ['ZF-BF', 'MMSE-BF', 'WMMSE']:
    ax.plot(pilot_snrs, csi_results[label], color=CMAP[label], marker=MMAP[label],
            markevery=2, alpha=0.7, label=label)
for label in all_models:
    ax.plot(pilot_snrs, csi_results[label], color=CMAP[label], marker=MMAP[label],
            markevery=2, linewidth=2.2, label=label)
ax.set_xlabel('Pilot SNR  ρₚ (dB)'); ax.set_ylabel('Sum-Rate (bps/Hz)')
ax.set_title(f'Sum-Rate vs. CSI Quality  —  Nt={Nt}, K={K}, data SNR={EVAL_REF_SNR:g} dB')
ax.legend(ncol=2); ax.grid(True)
finish_fig(fig, 'csi_quality.png')
save_plot_data('csi_quality', pilot_snr_dbs=pilot_snrs,
               **{k: np.array(v) for k, v in csi_results.items()})

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 11f. Params vs. sum-rate  +  validation curves of all trained models
# ══════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
for label, m in all_models.items():
    p, r = model_params(m) / 1e6, sr_results[label][idx20]
    ax.scatter(p, r, color=CMAP[label], s=150, marker=MMAP[label], zorder=3)
    ax.annotate(label, (p, r), textcoords='offset points', xytext=(6, 4), fontsize=9)
ax.set_xlabel('Parameters (M)'); ax.set_ylabel(f'Sum-Rate @ SNR={snr_dbs[idx20]:g} dB (bps/Hz)')
ax.set_title('Model Complexity vs. Performance'); ax.grid(True)

ax = axes[1]
for name, hist in all_histories.items():
    ax.plot([h['epoch'] for h in hist], [h['val_rate'] for h in hist],
            color=CMAP[name], label=name, linewidth=1.8)
ax.set_xlabel('Epoch (sum-rate phase)'); ax.set_ylabel('Val Sum-Rate (bps/Hz)')
ax.set_title('Training Curves — models trained in this run')
if all_histories: ax.legend()
ax.grid(True)
finish_fig(fig, 'complexity_and_curves.png')


# ══════════════════════════════════════════════════════════════════════
# 11g. Per-model training curve (train vs. val sum-rate, loss)
# ══════════════════════════════════════════════════════════════════════
for name, hist in all_histories.items():
    ep = np.array([h['epoch'] for h in hist])
    tl = np.array([h['train_loss'] for h in hist])
    vr = np.array([h['val_rate'] for h in hist])
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    ax1.plot(ep, -tl, color='#2980b9', label='Train Sum-Rate')
    ax1.plot(ep, vr,  color='#27ae60', label='Val Sum-Rate')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Sum-Rate (bps/Hz)')
    ax1.set_title(f'{name} — Training Curve'); ax1.legend(); ax1.grid(True)
    ax2.plot(ep, tl, color='#e74c3c')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss (neg. sum-rate)')
    ax2.set_title(f'{name} — Training Loss'); ax2.grid(True)
    finish_fig(fig, f'{name}_training_curve.png')
    save_plot_data(f'{name}_training_curve', epoch=ep, train_loss=tl, val_rate=vr)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 11h. Architecture ablation summary — BeamNet vs. its MLP replacements
# ══════════════════════════════════════════════════════════════════════
ABLATION_HDR = ['Model', 'Params', 'Val (bps/Hz)', 'Test (bps/Hz)', 'Δ Test vs BeamNet',
                'Sum-rate @ ref SNR', 'Δ vs BeamNet']
ablation_rows = []
_abl = [n for n in ['BeamNet'] + ABLATION_MODELS if n in all_models]
if any(n in all_models for n in ABLATION_MODELS):
    # Δ columns need BeamNet in this run (trained or loaded via PRETRAINED_CHECKPOINTS);
    # without it they show '—' and BeamNet's numbers from its own run are the reference.
    _has_bn = 'BeamNet' in all_models
    _bt = all_test_rates['BeamNet'] if _has_bn else None
    _bs = sr_results['BeamNet'][idx20] if _has_bn else None
    def _delta(x, ref, n):
        return '—' if (ref is None or n == 'BeamNet') else f'{x - ref:+.4f} ({(x - ref) / ref * 100:+.1f}%)'
    for n in _abl:
        t, s = all_test_rates[n], sr_results[n][idx20]
        ablation_rows.append({
            'Model': n, 'Params': f'{model_params(all_models[n]):,}',
            'Val (bps/Hz)': f'{all_val_rates[n]:.4f}', 'Test (bps/Hz)': f'{t:.4f}',
            'Δ Test vs BeamNet': _delta(t, _bt, n),
            'Sum-rate @ ref SNR': f'{s:.4f}',
            'Δ vs BeamNet': _delta(s, _bs, n)})

    print(f'Architecture ablation (Nt={Nt}, K={K}; ref SNR = {snr_dbs[idx20]:g} dB, '
          f'pilot SNR = {EVAL_PILOT_SNR:g} dB)\n')
    print(' | '.join(f'{h:<20}' for h in ABLATION_HDR))
    print('-' * (23 * len(ABLATION_HDR)))
    for r in ablation_rows:
        print(' | '.join(f'{r[h]:<20}' for h in ABLATION_HDR))
    with open(os.path.join(RESULTS_DIR, 'architecture_ablation.csv'), 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=ABLATION_HDR); w.writeheader(); w.writerows(ablation_rows)
    print('\n[Data] architecture_ablation.csv saved.')
    if not _has_bn:
        print('Note: BeamNet is not in this run, so the Δ columns are empty — compare with '
              'BeamNet\'s numbers from its own run (same dataset and evaluation seeds).')
else:
    print('[Skip] No architecture-ablation model (BeamNet-UserMLP / FlatMLP) in MODELS_TO_RUN.')

## 12 — Word report

One `.docx` combining both original reports: run metadata, training summary, model
summary (val/test), sum-rate at key SNR points, gain over MMSE-BF,
sample CSI / precoder matrices with per-user SINR (BeamNet vs. MMSE-BF),
full hyperparameters and references.

In [ ]:
def _ensure_python_docx():
    if importlib.util.find_spec('docx') is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                               'python-docx', '--break-system-packages', '-q'])

_ensure_python_docx()
from docx import Document as _Document
from docx.shared import Pt, RGBColor, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.ns import qn
from docx.oxml import OxmlElement


def _set_cell_bg(cell, hex_color):
    tcPr = cell._tc.get_or_add_tcPr()
    shd  = OxmlElement('w:shd')
    shd.set(qn('w:val'), 'clear'); shd.set(qn('w:color'), 'auto'); shd.set(qn('w:fill'), hex_color)
    tcPr.append(shd)

def _hdr_row(table, texts, bg='1F3864'):
    for cell, text in zip(table.rows[0].cells, texts):
        cell.text = ''
        run = cell.paragraphs[0].add_run(text)
        run.bold = True; run.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF); run.font.size = Pt(9)
        _set_cell_bg(cell, bg)

def _add_row(table, values, bold_first=False, alt=False):
    row = table.add_row()
    for j, val in enumerate(values):
        cell = row.cells[j]
        cell.text = str(val)
        run = cell.paragraphs[0].runs[0]
        run.font.size = Pt(9)
        if bold_first and j == 0: run.bold = True
        if alt: _set_cell_bg(cell, 'EBF3FB')

def _table(doc, header, rows, bold_first=True):
    t = doc.add_table(rows=1, cols=len(header)); t.style = 'Table Grid'
    _hdr_row(t, header)
    for i, r in enumerate(rows):
        _add_row(t, r, bold_first=bold_first, alt=(i % 2 == 1))
    doc.add_paragraph()
    return t

def _heading(doc, text, level=1):
    p = doc.add_heading(text, level=level)
    for run in p.runs: run.font.color.rgb = RGBColor(0x1F, 0x38, 0x64)
    return p

def _para(doc, text, size=10):
    p = doc.add_paragraph(text)
    p.runs[0].font.size = Pt(size)
    return p


def generate_report(n_report_samples=3):
    doc = _Document()
    for s in doc.sections:
        s.top_margin, s.bottom_margin, s.left_margin, s.right_margin = Cm(2), Cm(2), Cm(2.5), Cm(2)

    title = doc.add_heading(f'BeamTransformer — Run {RUN_ID} Report', 0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    for run in title.runs: run.font.color.rgb = RGBColor(0x1F, 0x38, 0x64)

    meta = [('Author',   AUTHOR),
            ('Course',   COURSE),
            ('Date',     datetime.datetime.now().strftime('%d %B, %Y')),
            ('Device',   DEVICE),
            ('System',   f'Nt={Nt}, K={K}, L={L}, {ARRAY}; eval pilot SNR={EVAL_PILOT_SNR:g} dB'),
            ('Dataset',  f'{N_TRAIN}/{N_VAL}/{N_TEST} train/val/test; SNR {SNR_RANGE} dB, '
                         f'pilot SNR {PILOT_SNR_RANGE} dB'),
            ('Training', f'AdamW lr={LR}, max_epochs={MAX_EPOCHS}, patience={PATIENCE}, '
                         f'WMMSE pre-train={PRETRAIN_EPOCHS} ep, batch={BATCH_SIZE}'),
            ('Models',   ', '.join(f'{n} ({model_params(m):,})' for n, m in all_models.items()))]
    t = doc.add_table(rows=len(meta), cols=2); t.style = 'Table Grid'
    for i, (k, v) in enumerate(meta):
        t.rows[i].cells[0].text, t.rows[i].cells[1].text = k, v
        t.rows[i].cells[0].paragraphs[0].runs[0].bold = True
        for c in t.rows[i].cells: c.paragraphs[0].runs[0].font.size = Pt(10)
        _set_cell_bg(t.rows[i].cells[0], 'D6E4F0')
    doc.add_paragraph()

    # 1. Training summary
    _heading(doc, '1. Training Summary')
    rows = []
    for name in all_models:
        hist = all_histories.get(name, [])
        rows.append([name, model_source[name],
                     str(hist[-1]['epoch']) if hist else '—',
                     f'{max(h["val_rate"] for h in hist):.4f}' if hist else '—',
                     f'{hist[-1]["train_loss"]:.4f}' if hist else '—'])
    _table(doc, ['Model', 'Source', 'Epochs', 'Best val (bps/Hz)', 'Final train loss'], rows)

    # 2. Model summary
    _heading(doc, '2. Model Summary (mean sum-rate over the val / test sets)')
    _table(doc, ['Model', 'Reference', 'Parameters', 'Val (bps/Hz)', 'Test (bps/Hz)'],
           [[n, MODEL_REFS[n], f'{model_params(m):,}', f'{all_val_rates[n]:.4f}',
             f'{all_test_rates[n]:.4f}'] for n, m in all_models.items()])

    # 3. Sum-rate at key SNR points
    _heading(doc, '3. Sum-Rate at Key SNR Points (bps/Hz)')
    key_snrs = [-5, 0, 5, 10, 15, 20, 25, 30, 35]
    cols = [l for l in ['MRT', 'ZF-BF', 'MMSE-BF', 'WMMSE'] + list(all_models) + ['Perfect CSI']
            if l in sr_results]
    rows = []
    for s in key_snrs:
        idx = int(np.argmin(np.abs(snr_dbs - s)))
        rows.append([f'{snr_dbs[idx]:g} dB'] + [f'{sr_results[l][idx]:.3f}' for l in cols])
    _table(doc, ['SNR'] + cols, rows)

    # 4. Gain over MMSE-BF
    _heading(doc, f'4. Gain over MMSE-BF at SNR = {snr_dbs[idx20]:g} dB')
    mmse_ref = sr_results['MMSE-BF'][idx20]
    rows = []
    for lbl in ['MRT', 'ZF-BF', 'MMSE-BF', 'WMMSE'] + list(all_models) + ['Perfect CSI']:
        r = sr_results[lbl][idx20]; diff = r - mmse_ref
        rows.append([lbl, f'{r:.4f}',
                     '—' if lbl == 'MMSE-BF' else f'{diff:+.4f}  ({diff / mmse_ref * 100:+.1f}%)'])
    _table(doc, ['Method', 'Sum-Rate (bps/Hz)', 'Gain vs MMSE-BF'], rows)

    # 4b. Architecture ablation
    if ablation_rows:
        _heading(doc, '4b. Architecture Ablation — Transformer vs. MLP')
        _para(doc, 'BeamNet-UserMLP = BeamNet with every self-attention sublayer removed (users '
                   'processed independently, SNR embedding added to each user token). FlatMLP = '
                   'fully-connected network on the whole flattened channel. Identical data, '
                   'WMMSE pre-training and sum-rate training for all models. '
                   f'Δ is relative to BeamNet; sum-rate columns at SNR = {snr_dbs[idx20]:g} dB '
                   f'and pilot SNR = {EVAL_PILOT_SNR:g} dB.')
        _table(doc, ABLATION_HDR, [[r[h] for h in ABLATION_HDR] for r in ablation_rows])

    # 5. Sample CSI & precoding matrices (BeamNet vs. MMSE-BF)
    _heading(doc, '5. Sample CSI & Precoding Matrices')
    if 'BeamNet' not in all_models:
        _para(doc, '[Skipped — BeamNet not in MODELS_TO_RUN]')
    else:
        bn = all_models['BeamNet']
        rng = np.random.default_rng(42)
        H_true_all = channel.generate(n_report_samples, rng=rng)
        H_hat_all  = add_csi_noise(H_true_all, EVAL_PILOT_SNR, rng=rng)
        for s_idx, snr_db in enumerate([5.0, 15.0, 25.0][:n_report_samples]):
            H_true, H_hat = H_true_all[s_idx], H_hat_all[s_idx]
            _heading(doc, f'Sample {s_idx + 1}  —  Data SNR = {snr_db:.0f} dB', level=2)

            ants = list(range(0, Nt, max(1, Nt // 8)))
            _heading(doc, '|H_hat[k, n]|  (subset of antennas)', level=3)
            _table(doc, ['User \\ Ant'] + [f'n={a}' for a in ants],
                   [[f'user {k}'] + [f'{np.abs(H_hat[k, a]):.4f}' for a in ants] for k in range(K)])

            W_mmse = mmse_bf(H_hat, snr_db)
            W_bn   = dl_precoders({'BeamNet': bn}, H_hat[None], snr_db, DEVICE)['BeamNet'][0]
            ants2  = list(range(0, Nt, max(1, Nt // 16)))
            for lbl, W in [('W_mmse', W_mmse), ('W_beamnet', W_bn)]:
                _heading(doc, f'|{lbl}[n, k]|  (subset of antennas)', level=3)
                _table(doc, ['Ant \\ User'] + [f'u{k}' for k in range(K)],
                       [[f'ant {a:>2}'] + [f'{np.abs(W[a, k]):.4f}' for k in range(K)] for a in ants2])
                p = doc.add_paragraph()
                r = p.add_run(f'||{lbl}||_F = {np.linalg.norm(W, "fro"):.6f}   '
                              f'(Total power = {np.linalg.norm(W, "fro") ** 2:.6f})')
                r.bold = True; r.font.size = Pt(10)

            _heading(doc, 'Per-User SINR (evaluated on true channel H)', level=3)
            _para(doc, 'SINR_k = |h_k^H w_k|^2 / (sum_{j≠k} |h_k^H w_j|^2 + sigma^2).', size=9)
            sinr_m, sinr_b = compute_sinr(H_true, W_mmse, snr_db), compute_sinr(H_true, W_bn, snr_db)
            rate_m, rate_b = np.log2(1 + sinr_m), np.log2(1 + sinr_b)
            rows = [[f'user {k}', f'{sinr_m[k]:.4f}', f'{rate_m[k]:.4f}',
                     f'{sinr_b[k]:.4f}', f'{rate_b[k]:.4f}', f'{rate_b[k] - rate_m[k]:+.4f}']
                    for k in range(K)]
            rows.append(['TOTAL', '—', f'{rate_m.sum():.4f}', '—', f'{rate_b.sum():.4f}',
                         f'{(rate_b - rate_m).sum():+.4f}'])
            _table(doc, ['User', 'SINR MMSE-BF', 'Rate MMSE-BF', 'SINR BeamNet',
                         'Rate BeamNet', 'Rate Gain'], rows)

    # 6. Hyperparameters
    _heading(doc, '6. Full Hyperparameter Record')
    _para(doc, 'A copy of all parameters is also saved as run_params.txt in the run folder.')
    params = [('Array type', ARRAY), ('Nt', Nt), ('K', K), ('L', L),
              ('N_TRAIN', N_TRAIN), ('N_VAL', N_VAL), ('N_TEST', N_TEST),
              ('BATCH_SIZE', BATCH_SIZE), ('SNR_RANGE', SNR_RANGE), ('PILOT_SNR_RANGE', PILOT_SNR_RANGE),
              ('WMMSE labels', f'n_iter={WMMSE_N_ITER}, tol={WMMSE_TOL}'),
              ('LR', LR), ('MAX_EPOCHS', MAX_EPOCHS), ('PATIENCE', PATIENCE),
              ('PRETRAIN_EPOCHS', f'{PRETRAIN_EPOCHS} (not for {", ".join(sorted(NO_PRETRAIN_MODELS))})'),
              ('BeamNet D_MODEL / N_LAYERS / N_HEADS', f'{D_MODEL} / {N_LAYERS} / {N_HEADS}'),
              ('IAIDNN layers', IAIDNN_LAYERS),
              ('UserMLP (abl. A) d_model / blocks', f'{USERMLP_D_MODEL} / {USERMLP_LAYERS}'),
              ('FlatMLP (abl. B) hidden / layers', f'{FLATMLP_HIDDEN} / {FLATMLP_LAYERS}'),
              ('FAST', FAST), ('SEED', SEED)]
    _table(doc, ['Parameter', 'Value'], [[k, str(v)] for k, v in params])

    # 7. References
    _heading(doc, '7. References')
    for ref in [
        '[B1] W. Xia, G. Zheng, Y. Zhu, J. Zhang, J. Wang, and A. P. Petropulu, "A deep learning '
        'framework for optimization of MISO downlink beamforming," IEEE Trans. Commun., vol. 68, '
        'no. 3, pp. 1866-1880, Mar. 2020.',
        '[B2] Q. Hu, Y. Cai, Q. Shi, K. Xu, G. Yu, and Z. Ding, "Iterative algorithm induced '
        'deep-unfolding neural networks: Precoding design for multiuser MIMO systems," IEEE Trans. '
        'Wireless Commun., vol. 20, no. 2, pp. 1394-1410, Feb. 2021. DOI: 10.1109/TWC.2020.3033334',
        '[B3] M. Zhang, J. Gao, and C. Zhong, "A deep learning-based framework for low complexity '
        'multiuser MIMO precoding design," IEEE Trans. Wireless Commun., vol. 21, no. 12, '
        'pp. 11193-11206, Dec. 2022. DOI: 10.1109/TWC.2022.3198060',
        '[W] Q. Shi, M. Razaviyayn, Z.-Q. Luo, and C. He, "An iteratively weighted MMSE approach to '
        'distributed sum-utility maximization for a MIMO interfering broadcast channel," IEEE Trans. '
        'Signal Process., vol. 59, no. 9, pp. 4331-4340, Sep. 2011.',
    ]:
        _para(doc, ref, size=9)

    path = os.path.join(RUN_DIR, f'Run_{RUN_ID}_Report.docx')
    doc.save(path)
    print(f'[Report] Saved to {path}')
    return path

report_path = generate_report()

## 13 — Display saved plots (optional)

Re-displays every PNG from this run's `results/` folder (useful after reconnecting).

In [ ]:
from IPython.display import Image, display

for fn in sorted(os.listdir(RESULTS_DIR)):
    if fn.endswith('.png'):
        print(f'\n── {fn} ──')
        display(Image(filename=os.path.join(RESULTS_DIR, fn)))

print(f'\n[Run {RUN_ID}] All results saved to: {RUN_DIR}')
print(f'  Plots      : {RESULTS_DIR}/')
print(f'  Plot data  : {RESULTS_DIR}/plot_data/   (npz + csv for MATLAB)')
print(f'  Checkpoints: {CHECKPOINT_DIR}/')
print(f'  Params     : {PARAMS_PATH}')
print(f'  Report     : {report_path}')